# Giai đoạn 5: Đánh giá cuối của model + phân tích protocol (CPU là đủ)

**5A. Đánh giá cuối bằng dự đoán trực tiếp.** Model cuối là `combo`: 3 encoder (main, seed1, seed2), chuẩn hoá theo session, trộn với LAG9. Trước đây các metric tính từ thống kê đủ mới chỉ có r và R². Ở đây model dự đoán **trực tiếp** các trial test để tính mọi metric:
- r theo dải tần, R²;
- `r_no50` (bỏ 46–54 Hz);
- `r_induced` (bỏ phần khoá theo task).

Ngoài ra có bảng **độ phủ – độ chính xác**: lọc contact theo r tính bằng K-fold trên calib (biết trước khi test), thì giữ được bao nhiêu % contact và r trên test là bao nhiêu. Trên dữ liệu giả, r từ dự đoán trực tiếp trùng khít r từ thống kê.

**5B. Phân tích protocol.** Cùng dữ liệu, cùng hai model (`base` và `enc`), chỉ thay protocol để đo mức thổi phồng:

| Protocol | Mô phỏng |
|---|---|
| `clean` | Của chúng ta |
| `rand_trials20` | Calib là 20% trial lấy **ngẫu nhiên**, xen kẽ với trial test |
| `rand_trials90` | Train 90% trial ngẫu nhiên, test 10% (kiểu NeuroFlowNet) |
| `rand_seg80` | Cắt thành đoạn 1 s, chia ngẫu nhiên 80/20, nên mẫu test nằm sát mẫu train (kiểu cửa sổ chồng lấn của CAST) |
| `mono_orig` | iEEG **monopolar** cộng EEG tham chiếu gốc (tham chiếu chung) |
| `mono_orig_seg80` | Monopolar + tham chiếu gốc + đoạn ngẫu nhiên (protocol dữ liệu kiểu CAST) |

| Metric | Ý nghĩa |
|---|---|
| `r_concat` | Của chúng ta: r trên toàn bộ phần test nối liền |
| `r_window` | r trong từng cửa sổ 1 s |
| `r_nfn` | Kiểu NeuroFlowNet: khôi phục biên độ bằng mean/std **của target** trong từng cửa sổ, rồi tính r trên cửa sổ đã làm phẳng mọi kênh |
| `r_nfn_null` | Metric `r_nfn` áp cho dự đoán **ngẫu nhiên**: metric "cho không" bao nhiêu |

Cuối cùng là các "công thức" mô phỏng tài liệu: kiểu CAST (thêm bước chỉ báo cáo trên bệnh nhân "viable", tức K-fold r > 0,2) và kiểu NeuroFlowNet.

Kiểm tra trên dữ liệu giả: `clean` tái tạo đúng kết quả của `readout`; trên dữ liệu giả có tham chiếu chung, monopolar thổi phồng r khoảng 25 lần.

Thời gian: 5A khoảng 40 phút, 5B khoảng 1 giờ trên CPU (nhanh hơn nếu có GPU). Cả hai chạy lại được nhiều lần, tiếp tục từ subject còn dở.

In [ ]:
#@title 1. Mount Drive + đường dẫn
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob
ROOT     = '/content/drive/MyDrive/iEEG_S2I'   #@param {type:"string"}
CODE_DIR = f'{ROOT}/code'
DATA_DIR = f'{ROOT}/data/s2i_v1'
RUNS_DIR = f'{ROOT}/runs'
MAIN_RUN = f'{RUNS_DIR}/shared_bip_car'
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)
os.environ['PYTHONPATH'] = CODE_DIR
os.chdir(CODE_DIR)
SEED_RUNS = sorted(r for r in glob.glob(f'{RUNS_DIR}/shared_seed*') if len(glob.glob(f'{r}/enc_sub-*.pt')) >= 15)
print('encoder seed:', [os.path.basename(r) for r in SEED_RUNS])

## 2. Ghi code lên Drive

In [ ]:
%%writefile {CODE_DIR}/s2i/nonlin.py
"""
Phase 1c (CPU) - can ANY non-linear EEG feature beat a well-regularised linear model with a
few minutes of calibration?

Phase 1b: the only linear lever that helped was better regularisation (ridge with one alpha
shared by all contacts, chosen by K-fold: +0.008 at 1-2 min, 15/15 subjects). Envelope
features raised alpha-band r by 17 % but the residual stage was selected by Pearson r, which
does not penalise amplitude, so it added noise (R^2 -> -0.16) and hurt small subjects.

Here the second stage is fitted on the OUT-OF-FOLD residual of the linear model and is
  * selected by MSE (not r), per contact, and
  * GATED: switched on for a contact only if its cross-validated MSE beats predicting the
    residual mean.
so it can only add what cross-validation says is real.

Models
------
  ridge         baseline of train_loso.py / ceiling.py (per-contact alpha, one 80/20 split)
  base          ridge_kfold_shared from Phase 1b (new strong baseline)
  env_gated     base + gated stage 2 on alpha/beta band waveforms + theta/alpha/beta envelopes
  wav_gated     base + gated stage 2 on per-channel Morlet envelopes, 12 log-spaced bands 2-64 Hz
                (exhaustive per-channel spectral power, incl. gamma)
  rocket_gated  base + gated stage 2 on random non-linear convolutional features
                (random spatial filter -> random Morlet wavelet -> envelope or ReLU).
                A cheap stand-in for "a non-linear encoder", with no training.
  all_gated     base + gated stage 2 on [wavelet envelopes, random features]

Usage
-----
python -m s2i.nonlin --data DATA_DIR --out RUNS_DIR/nonlin_bip_car
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
from scipy.fft import next_fast_len
from scipy.signal import butter, hilbert, sosfiltfilt

from s2i.baselines import lagged
from s2i.ceiling import StreamRidge, chrono_split
from s2i.metrics import per_contact_metrics
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge, _blocks

FS = 200
ENV_LAGS = (-20, -10, 0, 10, 20)
_B = {k: butter(4, f, btype="band", fs=FS, output="sos")
      for k, f in {"theta": (4, 8), "alpha": (8, 13), "beta": (13, 30)}.items()}


# ----------------------------------------------------------------------------- features
def feat_env(e):
    a = sosfiltfilt(_B["alpha"], e, axis=1)
    b = sosfiltfilt(_B["beta"], e, axis=1)
    t = sosfiltfilt(_B["theta"], e, axis=1)
    parts = [a, b, np.abs(hilbert(t, axis=1)), np.abs(hilbert(a, axis=1)), np.abs(hilbert(b, axis=1))]
    return lagged(np.concatenate(parts, 0).astype(np.float32), ENV_LAGS)


def morlet(freq, cycles):
    """Complex Morlet kernels (len(freq), 2H+1), unit L1 norm, centred at index H."""
    sig = np.asarray(cycles) / (2 * np.pi * np.asarray(freq))        # Gaussian width, s
    H = int(np.ceil(3 * sig.max() * FS))
    t = np.arange(-H, H + 1) / FS
    w = np.exp(-t[None] ** 2 / (2 * sig[:, None] ** 2)) * np.exp(2j * np.pi * np.asarray(freq)[:, None] * t[None])
    return (w / np.abs(w).sum(1, keepdims=True)).astype(np.complex64), H


def wconv(S, kern, H):
    """Zero-phase convolution of each row of S (M, L) with its own kernel (M, 2H+1) -> complex (M, L)."""
    L = S.shape[1]
    n = next_fast_len(L + 2 * H)
    return np.fft.ifft(np.fft.fft(S, n) * np.fft.fft(kern, n), n)[:, H:H + L]


class MorletEnvBank:
    """Per-channel Morlet envelope at n_freq log-spaced frequencies (all channels x all bands), lagged."""

    def __init__(self, freqs=tuple(np.geomspace(2, 64, 12)), cycles=5, lags=(-20, 0, 20)):
        self.kern1, self.H = morlet(freqs, np.full(len(freqs), cycles))
        self.nf, self.lags = len(freqs), lags

    def __call__(self, e):
        C = e.shape[0]
        S = np.repeat(e, self.nf, 0)
        env = np.abs(wconv(S, np.tile(self.kern1, (C, 1)), self.H))
        return lagged(env.astype(np.float32), self.lags)


class RandomConvFeatures:
    """
    M random (spatial filter -> complex Morlet wavelet -> non-linearity) features, fixed seed.
    A cheap stand-in for the first layer of a convolutional encoder, with no training:
      spatial filter : 1/3 single channel, 1/3 channel-pair difference, 1/3 dense random
      wavelet        : centre frequency log-uniform in [fmin, fmax] Hz, 3-7 cycles
      non-linearity  : half envelope |w * s|, half ReLU(Re(w * s) - b * rms), b ~ U(-1, 1)
    mode (same random filters/wavelets whatever the mode, so modes form a matched ablation):
      "mix" (above) | "env" all envelopes | "relu" all ReLU | "linear" Re(w * s), no non-linearity
    Synthetic check: dense spatial filters + short random 9-tap kernels (plain ROCKET) missed an
    alpha-envelope signal that feat_env recovered - broadband kernels dilute narrow-band power.
    """

    def __init__(self, n_chan, M=512, fmin=2.0, fmax=60.0, seed=0, mode="mix"):
        assert mode in ("mix", "env", "relu", "linear")
        self.mode = mode
        g = np.random.default_rng(seed)
        self.Ws = np.zeros((M, n_chan), np.float32)
        kind = np.arange(M) % 3
        for i in range(M):
            if kind[i] == 0 or n_chan < 2:
                self.Ws[i, g.integers(n_chan)] = 1
            elif kind[i] == 1:
                c = g.choice(n_chan, 2, replace=False)
                self.Ws[i, c] = (1, -1)
            else:
                self.Ws[i] = g.normal(size=n_chan)
        self.Ws /= np.linalg.norm(self.Ws, axis=1, keepdims=True)
        self.freq = np.exp(g.uniform(np.log(fmin), np.log(fmax), M))
        self.kern, self.H = morlet(self.freq, g.uniform(3, 7, M))
        self.env = g.random(M) < 0.5
        self.bias = g.uniform(-1, 1, M).astype(np.float32)

    def __call__(self, e):
        C = wconv(self.Ws @ e, self.kern, self.H)
        R = C.real
        if self.mode == "linear":
            return R.astype(np.float32)
        if self.mode == "env":
            return np.abs(C).astype(np.float32)
        out = np.maximum(R - self.bias[:, None] * np.sqrt((R * R).mean(1, keepdims=True)), 0)
        if self.mode == "mix":
            out[self.env] = np.abs(C[self.env])
        return out.astype(np.float32)


def concat_feats(*fs):
    return lambda e: np.concatenate([f(e) for f in fs], 0)


# ----------------------------------------------------------------------------- gated stage 2
class GatedResidualRidge:
    """
    Ridge on a residual target with per-contact alpha chosen by K-fold MSE, and a per-contact
    gate: ON only if the best CV MSE beats predicting the training-fold mean by `margin`.
    """

    def __init__(self, feat, alphas=ALPHAS_WIDE, k=5, margin=0.002):
        self.feat, self.alphas, self.k, self.margin = feat, alphas, k, margin
        self._sr = StreamRidge(feat, alphas)

    def fit(self, E, R):
        n, K = len(E), R[0].shape[0]
        folds = _blocks(n, self.k)
        fold_of = np.empty(n, int)
        for fi, idx in enumerate(folds):
            fold_of[idx] = fi
        st = [self._sr._stats([E[i] for i in idx], [R[i] for i in idx]) for idx in folds]
        total = [sum(x) for x in zip(*st)]
        models = [StreamRidge._solve([a - b for a, b in zip(total, s)]) for s in st]
        sse = np.zeros((len(self.alphas), K))
        sse_off = np.zeros(K)
        for i in range(n):
            s = models[fold_of[i]]
            t = R[i].T.astype(np.float64)
            Z = self._sr._design(s, E[i]) @ s["Q"]
            for ai, a in enumerate(self.alphas):
                p = Z @ (s["QB"] / (s["lam"] + a * s["scale"])[:, None]) + s["ym"]
                sse[ai] += ((p - t) ** 2).sum(0)
            sse_off += ((s["ym"] - t) ** 2).sum(0)
        best = np.argmin(sse, 0)
        self.gate = sse[best, np.arange(K)] < sse_off * (1 - self.margin)
        full = StreamRidge._solve(total)
        W = np.zeros((len(full["mu"]), K))
        for ai, a in enumerate(self.alphas):
            cols = np.where((best == ai) & self.gate)[0]
            if len(cols):
                W[:, cols] = full["Q"] @ (full["QB"][:, cols] / (full["lam"] + a * full["scale"])[:, None])
        self.s, self.W = full, W
        return self

    def predict(self, E):
        if not self.gate.any():
            return [np.zeros((len(self.gate), e.shape[1]), np.float32) for e in E]
        return [(self._sr._design(self.s, e) @ self.W + self.s["ym"] * self.gate).T.astype(np.float32) for e in E]


def gated(base, Ec, Yc, Et, feat):
    m = GatedResidualRidge(feat).fit(Ec, [y - o for y, o in zip(Yc, base.oof)])
    return [p + q for p, q in zip(base.predict(Et), m.predict(Et))], float(m.gate.mean())


# ----------------------------------------------------------------------------- per subject
def analyze(E, Y, a, log=print):
    n = len(E)
    _, test = chrono_split(n, 0.2, gap=1)
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    rfeat = RandomConvFeatures(E[0].shape[0], M=a.n_random, seed=a.seed)
    wfeat = MorletEnvBank()
    res = {"n_trials": n, "fracs": {}}
    for f in a.fracs:
        n_c = max(3, int(round(n * f)))
        if n_c >= test[0]:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        bands = abs(f - 0.2) < 1e-9
        out, on = {}, {}

        def score(name, P):
            m = per_contact_metrics(P, Yt, FS, bands=bands)
            out[name] = {"mean": {k: float(np.nanmean(v)) for k, v in m.items()}, "r": [float(x) for x in m["r"]]}

        score("ridge", StreamRidge(LAG9).fit(Ec, Yc).predict(Et))
        base = KFoldRidge(LAG9, shared=True).fit(Ec, Yc)
        score("base", base.predict(Et))
        P, on["env_gated"] = gated(base, Ec, Yc, Et, feat_env)
        score("env_gated", P)
        P, on["wav_gated"] = gated(base, Ec, Yc, Et, wfeat)
        score("wav_gated", P)
        P, on["rocket_gated"] = gated(base, Ec, Yc, Et, rfeat)
        score("rocket_gated", P)
        if not a.no_all:
            P, on["all_gated"] = gated(base, Ec, Yc, Et, concat_feats(wfeat, rfeat))
            score("all_gated", P)
        mins = sum(e.shape[1] for e in Ec) / FS / 60
        res["fracs"][f"{f:g}"] = {"n_calib": n_c, "minutes": mins, "gate_on": on,
                                  "mean": {k: v["mean"]["r"] for k, v in out.items()},
                                  "bands": {k: v["mean"] for k, v in out.items()} if bands else {},
                                  "r": {k: v["r"] for k, v in out.items()}}
        log(f"    f={f:g} ({mins:.1f} min) " + " ".join(f"{k}={v['mean']['r']:+.3f}" for k, v in out.items()) +
            " | gate on: " + " ".join(f"{k}={v:.2f}" for k, v in on.items()))
    return res


def summarize(results, out_dir, log=print):
    from scipy.stats import wilcoxon

    def wp(x, y):
        try:
            return float(wilcoxon(x, y).pvalue)
        except ValueError:
            return float("nan")

    lines, summ = [], {}
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r for r in results if f in r["fracs"]]
        models = list(rows[0]["fracs"][f]["mean"])
        M = {m: np.array([r["fracs"][f]["mean"][m] for r in rows]) for m in models}
        mins = np.mean([r["fracs"][f]["minutes"] for r in rows])
        lines += [f"\n### calib fraction {f}  ({len(rows)} subjects, mean {mins:.1f} min)",
                  "| model | mean r | vs ridge | p | vs base | wins vs base | p | gate on |",
                  "|---|---|---|---|---|---|---|---|"]
        summ[f] = {}
        for m in models:
            d0, d1 = M[m] - M["ridge"], M[m] - M["base"]
            gon = np.mean([r["fracs"][f]["gate_on"].get(m, np.nan) for r in rows])
            p0 = wp(M[m], M["ridge"]) if m != "ridge" else float("nan")
            p1 = wp(M[m], M["base"]) if m not in ("ridge", "base") else float("nan")
            summ[f][m] = {"mean": float(M[m].mean()), "diff_ridge": float(d0.mean()), "p_ridge": p0,
                          "diff_base": float(d1.mean()), "wins_base": int((d1 > 0).sum()), "p_base": p1,
                          "gate_on": float(gon)}
            lines.append(f"| {m} | {M[m].mean():.3f} | {d0.mean():+.4f} | {p0:.4f} | {d1.mean():+.4f} | "
                         f"{(d1 > 0).sum()}/{len(d1)} | {p1:.4f} | {gon:.2f} |")
    if "0.2" in fracs:
        rows = [r for r in results if "0.2" in r["fracs"]]
        models = list(rows[0]["fracs"]["0.2"]["bands"])
        keys = ["r", "r2", "r_delta", "r_theta", "r_alpha", "r_beta", "env_alpha", "env_beta", "env_gamma"]
        keys += [k for k in ("r_no50", "r_induced") if k in rows[0]["fracs"]["0.2"]["bands"][models[0]]]
        lines += ["\n### calib fraction 0.2: by band", "| model | " + " | ".join(keys) + " |",
                  "|---" * (len(keys) + 1) + "|"]
        for m in models:
            lines.append(f"| {m} | " + " | ".join(
                f"{np.nanmean([r['fracs']['0.2']['bands'][m].get(k, np.nan) for r in rows]):.3f}" for k in keys) + " |")
        models = list(rows[0]["fracs"]["0.2"]["mean"])
        lines += ["\n### per subject (calib fraction 0.2)", "| subject | " + " | ".join(models) + " |",
                  "|---" * (len(models) + 1) + "|"]
        for r in rows:
            lines.append(f"| {r['subject']} | " + " | ".join(f"{r['fracs']['0.2']['mean'][m]:.3f}" for m in models) + " |")
    md = "\n".join(lines)
    log(md)
    if out_dir:
        with open(os.path.join(out_dir, "summary.md"), "w", encoding="utf-8") as fh:
            fh.write(md + "\n")
        with open(os.path.join(out_dir, "summary.json"), "w") as fh:
            json.dump(summ, fh, indent=1)
    return summ


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--min_trials", type=int, default=20)
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2])
    ap.add_argument("--n_random", type=int, default=512)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no_all", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "nonlin.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    from s2i.ceiling import load_trials
    log(f"config: {vars(a)}")
    results = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        fp = os.path.join(a.out, f"nonlin_{sid}.json")
        if os.path.exists(fp) and not a.overwrite:
            results.append(json.load(open(fp)))
            log(f"[{sid}] already done, loaded")
            continue
        t0 = time.time()
        s, E, Y = load_trials(p, "bip", "car")
        if len(E) < a.min_trials or s.n_targets == 0:
            log(f"[{sid}] skipped")
            continue
        log(f"[{sid}] {len(E)} trials, {len(s.eeg_names)} scalp, {s.n_targets} targets")
        r = analyze(E, Y, a, log)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {time.time() - t0:.0f}s")
    if results:
        summarize(results, a.out, log)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/shared.py
"""
Phase 2 - shared temporal encoder + per-subject ridge decoder (LOSO, GPU).

Evidence from phases 0-1d (15 subjects, bipolar iEEG, CAR EEG, 20 % chronological calibration):
  * the SPATIAL scalp -> contact mapping does not transfer across subjects (1a: zero-shot r ~ 0);
  * within a subject the lever is frequency-selective, long-support LINEAR temporal filtering
    (1d: random Morlet filter bank, linear, gated on top of the ridge: +0.0075, 14/15 subjects),
    while non-linearity and more random features add ~nothing at 4 min of calibration.
Hypothesis: the TEMPORAL part (which filters of the scalp signal predict intracranial activity)
is shared across subjects even though the spatial part is not. Hence:
  encoder : per-channel filter bank, shared by all channels and all subjects, trained on the
            OTHER subjects. Quadrature pairs initialised as complex Morlet 2-60 Hz (+-1.2 s
            support) + the envelope of each pair + a few free filters.
  decoder : per-subject linear readout (channels x filters -> contacts), fitted in closed form by
            ridge on the calibration trials only, exactly like the baselines.
Training is episodic so it matches deployment: each step takes a training subject, fits a ridge
readout on ~4 min of consecutive trials (support), scores Pearson r on the following trials
(query), and back-propagates through the ridge solve into the filters. Model selection: the
same procedure on 2 held-out validation subjects (first 20 % of trials -> later trials).

Evaluation on the test subject (never seen in training; same split and calibration sizes as
phases 0-1d, so ridge / base / lin_gated must reproduce the phase-1d numbers):
  ridge, base, lin_gated  baselines (lin_gated = best model of phase 1d: the bar to beat)
  enc0_gated              base + gated stage 2 on the encoder AT INITIALISATION (untrained Morlet
                          bank): control - does the gain come from learning or from the init?
  lin_ridge, enc0_ridge   single-stage controls: ridge (K-fold, shared alpha) on the random linear
                          Morlet bank / on the UNTRAINED encoder
  enc_ridge               ridge (K-fold, shared alpha) on learned encoder features alone
  enc_ridge_cal           enc_ridge with a per-contact amplitude gain fitted on out-of-fold
                          calibration predictions (same r, honest R^2)
Every model is also scored with
  r_no50     46-54 Hz removed from prediction and target (the trained filters develop a sharp 50 Hz
             peak: mains residue after the notch filter);
  r_induced  each side minus its own trial average over the test trials, i.e. how well
             the SINGLE-TRIAL (non stimulus-locked) part is predicted.
  enc_gated               base + gated stage 2 on learned encoder features      <- main model

Usage
-----
python -m s2i.shared --data DATA_DIR --out RUNS_DIR/shared_bip_car
"""
import argparse
import copy
import glob
import json
import os
import sys
import time

import numpy as np
import torch
import torch.nn as nn
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import StreamRidge, chrono_split, load_trials, template, tmpl_at
from s2i.metrics import per_contact_metrics
from s2i.nonlin import RandomConvFeatures, gated, summarize
from s2i.within import ALPHAS_WIDE, LAG9, KFoldRidge

FS = 200
EXTRA_METRICS = ("r_no50", "r_induced")


# ----------------------------------------------------------------------------- encoder
class TemporalEncoder(nn.Module):
    """Per-channel filter bank shared by all channels and subjects: (B, C, L) -> (B, C, F, L)."""

    def __init__(self, n_pairs=12, half=240, fmin=2.0, fmax=60.0, cycles=5.0, n_free=4, envelope=True):
        super().__init__()
        self.cfg = dict(n_pairs=n_pairs, half=half, fmin=fmin, fmax=fmax, cycles=cycles, n_free=n_free,
                        envelope=envelope)
        self.n_pairs, self.envelope = n_pairs, envelope
        t = np.arange(-half, half + 1) / FS
        f = np.geomspace(fmin, fmax, n_pairs)
        gauss = np.exp(-t[None] ** 2 / (2 * (cycles / (2 * np.pi * f[:, None])) ** 2))
        w = np.zeros((2 * n_pairs + n_free, t.size))
        w[0:2 * n_pairs:2] = gauss * np.cos(2 * np.pi * f[:, None] * t[None])
        w[1:2 * n_pairs:2] = gauss * np.sin(2 * np.pi * f[:, None] * t[None])
        if n_free:
            w[2 * n_pairs, half] = 1.0                                   # identity (broadband, lag 0)
            g = np.random.default_rng(0)
            w[2 * n_pairs + 1:] = g.normal(size=(n_free - 1, t.size)) * np.exp(-t ** 2 / (2 * 0.05 ** 2))
        w /= np.linalg.norm(w, axis=1, keepdims=True)
        self.conv = nn.Conv1d(1, w.shape[0], 2 * half + 1, padding=half, bias=False)
        with torch.no_grad():
            self.conv.weight.copy_(torch.tensor(w, dtype=torch.float32)[:, None])

    @property
    def n_feat(self):
        return self.conv.out_channels + (self.n_pairs if self.envelope else 0)

    def forward(self, x):
        B, C, L = x.shape
        h = self.conv(x.reshape(B * C, 1, L))
        if self.envelope:
            p = h[:, :2 * self.n_pairs].float()
            h = torch.cat([h.float(), torch.sqrt(p[:, 0::2] ** 2 + p[:, 1::2] ** 2 + 1e-6)], 1)
        return h.reshape(B, C, -1, L)


# ----------------------------------------------------------------------------- torch helpers
def pad_trials(E, Y, idx, dev):
    L = max(E[i].shape[1] for i in idx)
    x = np.zeros((len(idx), E[idx[0]].shape[0], L), np.float32)
    y = np.zeros((len(idx), Y[idx[0]].shape[0], L), np.float32)
    m = np.zeros((len(idx), L), bool)
    for j, i in enumerate(idx):
        n = E[i].shape[1]
        x[j, :, :n], y[j, :, :n], m[j, :n] = E[i], Y[i], True
    return torch.from_numpy(x).to(dev), torch.from_numpy(y).to(dev), torch.from_numpy(m).to(dev)


def rows(enc, E, Y, idx, dev, amp, chunk=16):
    """Encoder features and targets of the given trials as (N, C*F) / (N, K) rows (padding removed)."""
    X, T = [], []
    for s in range(0, len(idx), chunk):
        x, y, m = pad_trials(E, Y, idx[s:s + chunk], dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)
        B, C, F, L = h.shape
        X.append(h.float().permute(0, 3, 1, 2).reshape(B, L, C * F)[m])
        T.append(y.permute(0, 2, 1)[m])
    return torch.cat(X), torch.cat(T)


def ridge_torch(Xs, Ys, Xq, lam):
    """Ridge on standardised features; lam is relative to the mean eigenvalue (= 1 after scaling)."""
    mu, sd = Xs.mean(0), Xs.std(0) + 1e-4
    A, Q = (Xs - mu) / sd, (Xq - mu) / sd
    ym = Ys.mean(0)
    N, D = A.shape
    G = A.T @ A / N + lam * torch.eye(D, device=A.device)
    return Q @ torch.linalg.solve(G, A.T @ (Ys - ym) / N) + ym


def col_r(P, T):
    P, T = P - P.mean(0), T - T.mean(0)
    return (P * T).sum(0) / ((P * P).sum(0).sqrt() * (T * T).sum(0).sqrt() + 1e-8)


def episode(n_trials, mean_len, rng, sup_sec, n_query):
    """Chronological episode: ~sup_sec of consecutive support trials, gap 1, then n_query trials."""
    n_q = min(n_query, max(1, n_trials // 4))
    n_s = int(np.clip(round(sup_sec * FS / mean_len), 3, n_trials - n_q - 1))
    start = int(rng.integers(0, n_trials - n_s - n_q - 1 + 1))
    return list(range(start, start + n_s)), list(range(start + n_s + 1, start + n_s + 1 + n_q))


@torch.no_grad()
def val_score(enc, lam, subjects, dev, amp, max_test):
    enc.eval()
    rs = []
    for _, E, Y in subjects:
        cal, test = chrono_split(len(E), 0.2, gap=1)
        Xs, Ys = rows(enc, E, Y, cal, dev, amp)
        Xq, Yq = rows(enc, E, Y, test[:max_test], dev, amp)
        rs.append(col_r(ridge_torch(Xs, Ys, Xq, lam), Yq).mean().item())
    return float(np.mean(rs))


# ----------------------------------------------------------------------------- training
def train_encoder(a, train, val, dev, log, ckpt_path):
    enc = TemporalEncoder(**enc_cfg(a)).to(dev)
    log_lam = nn.Parameter(torch.tensor(float(np.log(a.lam0)), device=dev))
    opt = torch.optim.AdamW([{"params": enc.parameters(), "lr": a.lr, "weight_decay": 0.0},
                             {"params": [log_lam], "lr": a.lr_lam, "weight_decay": 0.0}])
    amp = a.amp and dev.type == "cuda"
    rng = np.random.default_rng(a.seed)
    mean_len = {sid: float(np.mean([e.shape[1] for e in E])) for sid, E, _ in train}
    step, best, best_state, bad, hist = 0, -1e9, None, 0, []
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=dev, weights_only=False)
        enc.load_state_dict(ck["enc"])
        opt.load_state_dict(ck["opt"])
        with torch.no_grad():
            log_lam.copy_(ck["log_lam"])
        step, best, best_state, bad, hist = ck["step"], ck["best"], ck["best_state"], ck["bad"], ck["hist"]
        rng = np.random.default_rng(a.seed + step)
        log(f"  resumed at step {step} (best val r={best:.4f})")
    else:
        r0 = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
        hist.append((0, r0))
        log(f"  step     0 | val r (Morlet init) = {r0:.4f}")
        best, best_state = r0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
    t0, run_r = time.time(), []
    while step < a.steps and bad < a.patience:
        enc.train()
        loss = 0.0
        for si in rng.choice(len(train), a.subj_per_step, replace=len(train) < a.subj_per_step):
            sid, E, Y = train[si]
            sup, qry = episode(len(E), mean_len[sid], rng, a.sup_sec, a.n_query)
            Xs, Ys = rows(enc, E, Y, sup, dev, amp)
            Xq, Yq = rows(enc, E, Y, qry, dev, amp)
            r = col_r(ridge_torch(Xs, Ys, Xq, log_lam.exp()), Yq).mean()
            loss = loss + (1 - r) / a.subj_per_step
            run_r.append(r.item())
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
        opt.step()
        step += 1
        if step % a.eval_every == 0 or step == a.steps:
            vr = val_score(enc, log_lam.exp().item(), val, dev, amp, a.val_trials)
            hist.append((step, vr))
            if vr > best:
                best, bad, best_state = vr, 0, (copy.deepcopy(enc.state_dict()), log_lam.exp().item())
            else:
                bad += 1
            log(f"  step {step:5d} | train query r={np.mean(run_r):.4f} | val r={vr:.4f} | "
                f"lam={log_lam.exp().item():.3g} | {time.time() - t0:.0f}s" + (" *" if bad == 0 else ""))
            run_r, t0 = [], time.time()
            if ckpt_path:
                torch.save({"enc": enc.state_dict(), "opt": opt.state_dict(), "log_lam": log_lam.detach(),
                            "step": step, "best": best, "best_state": best_state, "bad": bad, "hist": hist},
                           ckpt_path + ".tmp")
                os.replace(ckpt_path + ".tmp", ckpt_path)
    enc.load_state_dict(best_state[0])
    return enc, best_state[1], best, hist


def enc_cfg(a):
    return dict(n_pairs=a.n_pairs, half=a.half, fmin=a.fmin, fmax=a.fmax, cycles=a.cycles,
                n_free=a.n_free, envelope=not a.no_env)


# ----------------------------------------------------------------------------- evaluation
def np_feat(enc, dev, amp):
    """Encoder as a numpy feature function (C, L) -> (C*F, L) for the StreamRidge machinery."""
    enc = enc.eval()

    @torch.no_grad()
    def f(e):
        x = torch.from_numpy(np.ascontiguousarray(e, np.float32))[None].to(dev)
        with torch.autocast(device_type=dev.type, enabled=amp):
            h = enc(x)[0]
        return h.float().reshape(-1, h.shape[-1]).cpu().numpy()
    return f


_LINE = butter(4, (46, 54), btype="bandstop", fs=FS, output="sos")


def r_without_line(P, T):
    """Per-contact r after removing 46-54 Hz (mains residue) from BOTH prediction and target."""
    fp = [sosfiltfilt(_LINE, p, axis=1) for p in P]
    ft = [sosfiltfilt(_LINE, t, axis=1) for t in T]
    return per_contact_metrics(fp, ft, FS, bands=False)["r"]


def r_single_trial(P, T):
    """
    Per-contact r of the single-trial parts: each side minus ITS OWN trial average over the test
    trials (prediction minus the mean prediction, target minus the mean target). Removing the same
    template from both sides would add a shared component and inflate r.
    """
    tP, tT = template(P), template(T)
    return per_contact_metrics([p - tmpl_at(tP, p.shape[1]) for p in P],
                               [t - tmpl_at(tT, t.shape[1]) for t in T], FS, bands=False)["r"]


def amplitude_calibrate(kf, Yc, P):
    """
    Per-contact least-squares gain on the OUT-OF-FOLD calibration predictions. Alpha is chosen by
    r, which ignores amplitude; this fixes the amplitude (R^2) without changing r (gain >= 0).
    """
    ym = kf.s["ym"][:, None]
    num = sum(((o - ym) * (y - ym)).sum(1) for o, y in zip(kf.oof, Yc))
    den = sum(((o - ym) ** 2).sum(1) for o in kf.oof)
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)[:, None]
    return [(ym + g * (p - ym)).astype(np.float32) for p in P]


def evaluate(E, Y, enc0, enc, a, dev, log):
    amp = a.amp and dev.type == "cuda"
    n = len(E)
    _, test = chrono_split(n, 0.2, gap=1)
    Et, Yt = [E[i] for i in test], [Y[i] for i in test]
    f0, f1 = np_feat(enc0, dev, amp), np_feat(enc, dev, amp)
    lin = RandomConvFeatures(E[0].shape[0], M=512, seed=0, mode="linear")
    res = {"n_trials": n, "fracs": {}}
    for f in a.fracs:
        n_c = max(3, int(round(n * f)))
        if n_c >= test[0]:
            continue
        Ec, Yc = E[:n_c], Y[:n_c]
        bands = abs(f - 0.2) < 1e-9
        out, on = {}, {}

        def score(name, P):
            m = per_contact_metrics(P, Yt, FS, bands=bands)
            m["r_no50"] = r_without_line(P, Yt)
            m["r_induced"] = r_single_trial(P, Yt)
            out[name] = {"mean": {k: float(np.nanmean(v)) for k, v in m.items()}, "r": [float(x) for x in m["r"]],
                         **{x: [float(v) for v in m[x]] for x in EXTRA_METRICS}}

        score("ridge", StreamRidge(LAG9).fit(Ec, Yc).predict(Et))
        base = KFoldRidge(LAG9, shared=True).fit(Ec, Yc)
        score("base", base.predict(Et))
        if not a.fast_eval:
            for name, feat in [("lin_gated", lin), ("enc0_gated", f0), ("enc_gated", f1)]:
                P, on[name] = gated(base, Ec, Yc, Et, feat)
                score(name, P)
        for name, feat in [("lin_ridge", lin)] * (not a.fast_eval) + [("enc0_ridge", f0)]:
            score(name, KFoldRidge(feat, ALPHAS_WIDE, shared=True, keep_oof=False).fit(Ec, Yc).predict(Et))
        kf = KFoldRidge(f1, ALPHAS_WIDE, shared=True).fit(Ec, Yc)
        P = kf.predict(Et)
        score("enc_ridge", P)
        score("enc_ridge_cal", amplitude_calibrate(kf, Yc, P))
        mins = sum(e.shape[1] for e in Ec) / FS / 60
        res["fracs"][f"{f:g}"] = {"n_calib": n_c, "minutes": mins, "gate_on": on,
                                  "mean": {k: v["mean"]["r"] for k, v in out.items()},
                                  **{x: {k: v[x] for k, v in out.items()} for x in EXTRA_METRICS},
                                  "bands": {k: v["mean"] for k, v in out.items()} if bands else {},
                                  "r": {k: v["r"] for k, v in out.items()}}
        log(f"    f={f:g} ({mins:.1f} min) " + " ".join(f"{k}={v['mean']['r']:+.3f}" for k, v in out.items()) +
            " | gate on: " + " ".join(f"{k}={v:.2f}" for k, v in on.items()))
    return res


KEY_TESTS = [("enc_ridge", "enc0_ridge", "learning helps (vs same encoder untrained)?"),
             ("enc_ridge", "lin_ridge", "learned filters beat random filters?"),
             ("enc0_ridge", "lin_ridge", "Morlet init beats random filters?"),
             ("enc_ridge", "base", "beats the strong linear baseline?"),
             ("lin_ridge", "base", "single-stage filter bank beats LAG9?")]


def key_tests(results, out_dir, log):
    """Paired subject-level tests for the decisive comparisons, on r and on every EXTRA_METRICS."""
    from scipy.stats import wilcoxon
    lines, js = ["\n### key comparisons (paired over subjects)",
                 "| calib | question | A | B | metric | mean A | mean B | A - B | A wins | Wilcoxon p |",
                 "|---|---|---|---|---|---|---|---|---|---|"], []
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r for r in results if f in r["fracs"] and "r_no50" in r["fracs"][f]]
        metrics = ("r",) + tuple(x for x in EXTRA_METRICS if rows and x in rows[0]["fracs"][f])
        for ma, mb, q in KEY_TESTS:
            if not rows or ma not in rows[0]["fracs"][f]["mean"] or mb not in rows[0]["fracs"][f]["mean"]:
                continue
            for metric in metrics:
                if metric == "r":
                    get = lambda r, m: r["fracs"][f]["mean"][m]                         # noqa: E731
                else:
                    get = lambda r, m: float(np.nanmean(r["fracs"][f][metric][m]))     # noqa: E731
                x = np.array([get(r, ma) for r in rows])
                y = np.array([get(r, mb) for r in rows])
                try:
                    p = float(wilcoxon(x, y).pvalue)
                except ValueError:
                    p = float("nan")
                js.append({"frac": f, "a": ma, "b": mb, "metric": metric, "mean_a": float(x.mean()),
                           "mean_b": float(y.mean()), "diff": float((x - y).mean()), "wins": int((x > y).sum()),
                           "n": len(x), "p": p})
                lines.append(f"| {f} | {q} | {ma} | {mb} | {metric} | {x.mean():.3f} | {y.mean():.3f} | "
                             f"{(x - y).mean():+.4f} | {(x > y).sum()}/{len(x)} | {p:.4f} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "summary.md"), "a", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "key_tests.json"), "w") as fh:
        json.dump(js, fh, indent=1)


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--subjects", nargs="*", default=None, help="subset of subjects to load")
    ap.add_argument("--folds", nargs="*", default=None, help="test subjects to run (default: all)")
    ap.add_argument("--min_trials", type=int, default=20)
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.1, 0.2])
    # encoder
    ap.add_argument("--n_pairs", type=int, default=12)
    ap.add_argument("--half", type=int, default=240, help="filter half-length in samples (240 = 1.2 s)")
    ap.add_argument("--fmin", type=float, default=2.0)
    ap.add_argument("--fmax", type=float, default=60.0)
    ap.add_argument("--cycles", type=float, default=5.0)
    ap.add_argument("--n_free", type=int, default=4)
    ap.add_argument("--no_env", action="store_true")
    # episodic training
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--eval_every", type=int, default=150)
    ap.add_argument("--patience", type=int, default=6)
    ap.add_argument("--subj_per_step", type=int, default=2)
    ap.add_argument("--sup_sec", type=float, default=240.0, help="support size (seconds), ~4 min as at test")
    ap.add_argument("--n_query", type=int, default=8)
    ap.add_argument("--lr", type=float, default=3e-4)
    ap.add_argument("--lr_lam", type=float, default=1e-2)
    ap.add_argument("--lam0", type=float, default=1.0)
    ap.add_argument("--n_val_subj", type=int, default=2)
    ap.add_argument("--val_trials", type=int, default=40, help="test trials per validation subject")
    ap.add_argument("--no_amp", action="store_true")
    ap.add_argument("--fast_eval", action="store_true",
                    help="only ridge, base, enc0_ridge, enc_ridge, enc_ridge_cal (for seed / ablation runs)")
    ap.add_argument("--reeval", action="store_true",
                    help="redo the evaluation of finished folds with their saved encoders (no retraining)")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    a.amp = not a.no_amp
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "shared.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    log(f"config: {vars(a)}")
    subjects = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        sid = os.path.basename(p)[:-3]
        if a.subjects and sid not in a.subjects:
            continue
        s, E, Y = load_trials(p, "bip", "car")
        if len(E) < a.min_trials or s.n_targets == 0:
            log(f"skip {sid}: {len(E)} trials, {s.n_targets} targets")
            continue
        subjects.append((sid, E, Y))
        log(f"{sid}: {len(E)} trials, {E[0].shape[0]} scalp, {s.n_targets} targets")
    folds = [s for s in subjects if not a.folds or s[0] in a.folds]

    results = []
    for fi, (sid, E, Y) in enumerate(folds):
        fp = os.path.join(a.out, f"fold_{sid}.json")
        if os.path.exists(fp):
            done = json.load(open(fp))
            # --reeval redoes only folds evaluated by an older version (missing a metric), so it can resume
            if not a.reeval or all(x in fr for fr in done["fracs"].values() for x in EXTRA_METRICS):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
        torch.manual_seed(a.seed + fi)
        log("=" * 78)
        log(f"[{fi + 1}/{len(folds)}] TEST {sid}")
        pool = [s for s in subjects if s[0] != sid]
        order = np.random.default_rng(a.seed + fi).permutation(len(pool))
        val = [pool[i] for i in order[:a.n_val_subj]]
        train = [pool[i] for i in order[a.n_val_subj:]]
        log(f"  train={[s[0] for s in train]} val={[s[0] for s in val]}")
        t0 = time.time()
        enc_path = os.path.join(a.out, f"enc_{sid}.pt")
        if os.path.exists(enc_path):
            ck = torch.load(enc_path, map_location=dev, weights_only=False)
            enc = TemporalEncoder(**ck["cfg"]).to(dev)
            enc.load_state_dict(ck["state"])
            lam, best, hist = ck["lam"], ck["best_val"], ck["hist"]
            log(f"  loaded trained encoder (val r={best:.4f})")
        else:
            ckpt = os.path.join(a.out, f"ckpt_{sid}.pt")
            enc, lam, best, hist = train_encoder(a, train, val, dev, log, ckpt)
            torch.save({"state": enc.state_dict(), "cfg": enc.cfg, "lam": lam, "best_val": best, "hist": hist},
                       enc_path)
            if os.path.exists(ckpt):
                os.remove(ckpt)
        enc0 = TemporalEncoder(**enc.cfg).to(dev)
        log(f"  encoder: val r {hist[0][1]:.4f} (init) -> {best:.4f} (trained), lam={lam:.3g}, "
            f"{enc.n_feat} features/channel | {time.time() - t0:.0f}s")
        r = evaluate(E, Y, enc0, enc, a, dev, log)
        r.update(subject=sid, val_subjects=[s[0] for s in val], val_r_init=hist[0][1], val_r=best, lam=lam,
                 minutes=(time.time() - t0) / 60)
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
        del enc, enc0
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    if results:
        summarize(results, a.out, log)
        key_tests(results, a.out, log)
        vi = [r["val_r_init"] for r in results]
        vt = [r["val_r"] for r in results]
        log(f"validation subjects, ridge readout on 20 %: Morlet init r={np.mean(vi):.4f} -> trained {np.mean(vt):.4f}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/readout.py
"""
Phase 4 - readout improvements on the SAVED shared encoders (CPU, no retraining).

Phase 3 analysis of the shared encoder (15 subjects, 20 % calibration):
  (1) training traded high frequencies for low ones: the untrained Morlet bank (enc0) beats the
      trained encoder in alpha/beta, the trained one wins in delta/theta; two pairs collapsed on 50 Hz;
  (2) half of the features (pairs >= 13 Hz and their envelopes) sit in bands with little predictable
      signal, but one shared ridge penalty treats them like the useful ones;
  (3) at 1-2 min of calibration the readout (C x 40 weights per contact) is the bottleneck.
Every candidate below keeps the encoder fixed and changes only the readout:

  enc         reference = enc_ridge of s2i.shared (ridge, one K-fold alpha)
  base        reference = ridge_kfold_shared on LAG9
  enc_low     only low-frequency encoder features (pairs < 13 Hz, their envelopes, free filters)
  enc_group   high-frequency features down-weighted by a K-fold-chosen factor (two penalties)
  enc_enc0hi  + the UNTRAINED encoder's high-frequency features (alpha/beta), own K-fold weight
  enc_lag9    + LAG9 (raw lags +-200 ms), own K-fold weight
  enc_bands   target split into 5 complementary bands (<4, 4-8, 8-13, 13-30, >30 Hz); one ridge per
              band with its own K-fold penalty (chosen by MSE); predictions summed
  enc_sess    encoder features standardised per recording session (EEG only, label-free,
              transductive: uses the EEG of every trial of the session)
  enc_warp    filters time-scaled by s in {0.9, 1, 1.1} (frequencies x 1/s), s chosen per subject by
              K-fold (individual frequency adaptation)
  enc_bilin   structured readout W_k = a_k b_k^T (channels x features, rank 1-3, spectral profiles b
              orthonormal, ridge penalty on a), rank and penalty by K-fold: (C + F) R weights per contact
  ens_base    enc and base averaged with a K-fold-chosen weight
  ens_seed    enc of this run and of the seed-1 run averaged with a K-fold-chosen weight (if present)
--strict: undo preprocessing's whole-session normalisation and re-normalise EEG and target with the
calibration trials' statistics only; session-standardised models are skipped (combo_nosess = combo
without session standardisation is reported in every mode).
Phase 4b (--only_combo, combinations of the three Phase-4 winners):
  ens_all       ridge readouts of ALL available encoders (main + seed runs) averaged
  ens_all_sess  same, every encoder's features standardised per session
  combo         ens_all_sess mixed with base (LAG9 ridge), weight chosen by K-fold

Exactness: every readout is linear in its features, so K-fold out-of-fold r, the amplitude gain and
test r / R^2 are computed EXACTLY from second-order sufficient statistics (sums of x, xx', y, y^2,
xy') accumulated in one pass over the trials. `enc` therefore reproduces enc_ridge of s2i.shared.
Test trials, calibration fractions and fold blocks are those of phases 0-3.

Usage
-----
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car [--seed_run RUNS_DIR/shared_seed1]
python -m s2i.readout --data DATA_DIR --run RUNS_DIR/shared_bip_car --only_combo
       --seed_runs RUNS_DIR/shared_seed1 RUNS_DIR/shared_seed2     (Phase 4b: combinations of the winners)
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import torch
from scipy.signal import butter, sosfiltfilt

from s2i.ceiling import ALPHAS, chrono_split, feat_lag
from s2i.baselines import DEFAULT_LAGS
from s2i.metrics import per_contact_metrics
from s2i.shared import TemporalEncoder, np_feat, r_single_trial, r_without_line
from s2i.within import ALPHAS_WIDE, _blocks

FS = 200
HIGH_HZ = 13.0
Y_CUTS = (4.0, 8.0, 13.0, 30.0)                       # -> 5 complementary target bands
GROUP_SCALES = (1.0, 0.5, 0.25, 0.1, 0.0)             # weight of the high-frequency group
ADD_SCALES = (0.0, 0.25, 0.5, 1.0, 2.0)               # weight of an added feature block
ENS_W = (0.0, 0.25, 0.5, 0.75, 1.0)                   # weight of enc in an ensemble
WARPS = (0.9, 1.1)
BILIN = tuple((r, al) for r in (1, 2, 3) for al in (0.001, 0.01, 0.1, 1.0, 10.0))
LAG9 = feat_lag(DEFAULT_LAGS)
_LP = {c: butter(4, c, btype="low", fs=FS, output="sos") for c in Y_CUTS}
_DEV = None                                           # set in main(); Gram matrices on the GPU if cuda


# ----------------------------------------------------------------------------- statistics
class St:
    """Second-order sufficient statistics of (features x, target y); optional band targets."""

    def __init__(self, D, K, nb=0, Db=0):
        self.n = 0.0
        self.sx, self.sxx = np.zeros(D), np.zeros((D, D))
        self.sy, self.syy, self.sxy = np.zeros(K), np.zeros(K), np.zeros((D, K))
        self.syb, self.syyb, self.sxyb = np.zeros((nb, K)), np.zeros((nb, K)), np.zeros((nb, Db, K))

    def add(self, X, y, yb=None):
        self.n += X.shape[1]
        self.sx += X.sum(1)
        if _DEV is not None and _DEV.type == "cuda":             # large Gram matrices on the GPU
            Xt, yt = torch.from_numpy(X).to(_DEV), torch.from_numpy(np.ascontiguousarray(y)).to(_DEV)
            self.sxx += (Xt @ Xt.T).double().cpu().numpy()
            self.sxy += (Xt @ yt.T).double().cpu().numpy()
        else:
            self.sxx += X @ X.T
            self.sxy += X @ y.T
        self.sy += y.sum(1)
        self.syy += (y * y).sum(1)
        if yb is not None:
            Xb = X[:self.sxyb.shape[1]]
            self.syb += yb.sum(2)
            self.syyb += (yb * yb).sum(2)
            for b in range(len(yb)):
                self.sxyb[b] += Xb @ yb[b].T

    def _new(self):
        return St.__new__(St)

    def __add__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) + getattr(o, k))
        return s

    def __sub__(self, o):
        s = self._new()
        for k in ("n", "sx", "sxx", "sy", "syy", "sxy", "syb", "syyb", "sxyb"):
            setattr(s, k, getattr(self, k) - getattr(o, k))
        return s

    def take(self, idx, band=None, transform=None):
        """Sub-block of features `idx` (band b replaces the target), optionally x -> (x - m) / d."""
        s = self._new()
        s.n, sx, sxx = self.n, self.sx[idx], self.sxx[np.ix_(idx, idx)]
        if band is None:
            s.sy, s.syy, sxy = self.sy, self.syy, self.sxy[idx]
        else:
            s.sy, s.syy, sxy = self.syb[band], self.syyb[band], self.sxyb[band][idx]
        if transform is not None:
            m, d = transform
            sxx = (sxx - np.outer(m, sx) - np.outer(sx, m) + s.n * np.outer(m, m)) / np.outer(d, d)
            sxy = (sxy - np.outer(m, s.sy)) / d[:, None]
            sx = (sx - s.n * m) / d
        s.sx, s.sxx, s.sxy = sx, sxx, sxy
        K = len(s.sy)
        s.syb, s.syyb, s.sxyb = np.zeros((0, K)), np.zeros((0, K)), np.zeros((0, len(idx), K))
        return s


def total(sts):
    out = sts[0]
    for s in sts[1:]:
        out = out + s
    return out


# ----------------------------------------------------------------------------- linear readouts
def standardise(tr):
    n = tr.n
    mu, ym = tr.sx / n, tr.sy / n
    G = tr.sxx - n * np.outer(mu, mu)
    sd = np.sqrt(np.clip(np.diag(G) / n, 0, None)) + 1e-8
    Gs = G / np.outer(sd, sd)
    Bs = (tr.sxy - n * np.outer(mu, ym)) / sd[:, None]
    return mu, sd, (Gs + Gs.T) / 2, Bs, ym


def to_raw(Ws, mu, sd, ym):
    A = Ws / sd[:, None]
    return A, ym - mu @ A


def ridge_models(tr, alphas, scale_vec=None):
    """Ridge for each alpha (penalty alpha * mean eigenvalue). scale_vec multiplies standardised features."""
    mu, sd, Gs, Bs, ym = standardise(tr)
    sc = max(np.trace(Gs) / len(Gs), 1e-12)      # penalty scale of the UNSCALED design (= n)
    if scale_vec is not None:
        Gs, Bs = Gs * np.outer(scale_vec, scale_vec), Bs * scale_vec[:, None]
    lam, Q = np.linalg.eigh(Gs)
    lam = np.clip(lam, 0, None)
    QB = Q.T @ Bs
    out = []
    for a in alphas:
        Ws = Q @ (QB / (lam + a * sc)[:, None])
        if scale_vec is not None:
            Ws = Ws * scale_vec[:, None]
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def bilinear_models(tr, C, F, cands, iters=5):
    """
    W_k = a_k b_k^T over (channel, feature) with rank R: b_k (F x R) has orthonormal columns (spectral
    profiles), a_k (C x R) carries the ridge penalty alpha * n, exactly the ridge scale. Alternating
    least squares, initialised by the SVD of a ridge fit.
    """
    mu, sd, Gs, Bs, ym = standardise(tr)
    K, sc = Bs.shape[1], max(np.trace(Gs) / len(Gs), 1e-12)
    G4, B3 = Gs.reshape(C, F, C, F), Bs.reshape(C, F, K)
    lam, Q = np.linalg.eigh(Gs)
    W0 = Q @ ((Q.T @ Bs) / (np.clip(lam, 0, None) + sc)[:, None])
    _, _, Vt = np.linalg.svd(W0.T.reshape(K, C, F), full_matrices=False)
    out = []
    for R, alpha in cands:
        R_ = min(R, Vt.shape[1])
        b = Vt[:, :R_, :].transpose(0, 2, 1)                     # (K, F, R) orthonormal columns

        def solve_a(b):
            # Ga[k] = (I_C x b_k)' G (I_C x b_k): contract g with b (one matmul), then f (batched matmul)
            M = (Gs.reshape(C * F * C, F) @ b.transpose(1, 0, 2).reshape(F, K * R_)).reshape(C, F, C, K, R_)
            Ga = np.matmul(M.transpose(3, 0, 2, 4, 1), b[:, None, None]).transpose(0, 1, 4, 2, 3)
            ca = np.einsum("cfk,kfr->kcr", B3, b).reshape(K, C * R_, 1)
            return np.linalg.solve(Ga.reshape(K, C * R_, C * R_) + alpha * sc * np.eye(C * R_),
                                   ca)[..., 0].reshape(K, C, R_)
        a = solve_a(b)
        for _ in range(iters):
            # Gb[k] = (a_k x I_F)' G (a_k x I_F): contract c with a (one matmul), then d (batched matmul)
            M = (a.transpose(0, 2, 1).reshape(K * R_, C) @ Gs.reshape(C, F * C * F)).reshape(K, R_, F, C, F)
            Gb = np.matmul(M.transpose(0, 1, 2, 4, 3), a[:, None, None]).transpose(0, 2, 1, 3, 4)
            Gb = Gb.reshape(K, F * R_, F * R_)
            cb = np.einsum("cfk,kcr->kfr", B3, a).reshape(K, F * R_, 1)
            b = np.linalg.solve(Gb + 1e-4 * sc * np.eye(F * R_), cb)[..., 0].reshape(K, F, R_)
            b, _ = np.linalg.qr(b)                               # keep only the profile subspace
            a = solve_a(b)
        Ws = np.einsum("kcr,kfr->cfk", a, b).reshape(C * F, K)
        out.append(to_raw(Ws, mu, sd, ym))
    return out


def pred_sums(A, c, te):
    """Sums of prediction p = x'A + c over the samples summarised by `te`: (n, sp, spp, spy, sy, syy)."""
    Ax = A.T @ te.sx
    sp = Ax + te.n * c
    spp = ((te.sxx @ A) * A).sum(0) + 2 * c * Ax + te.n * c ** 2
    spy = (A * te.sxy).sum(0) + c * te.sy
    return np.stack([np.full_like(sp, te.n), sp, spp, spy, te.sy, te.syy])


def r_of(s):
    n, sp, spp, spy, sy, syy = s
    den = np.sqrt(np.clip(n * spp - sp ** 2, 0, None) * np.clip(n * syy - sy ** 2, 0, None))
    return np.where(den > 0, (n * spy - sp * sy) / np.maximum(den, 1e-12), np.nan)


def sse_of(s):
    return s[2] - 2 * s[3] + s[5]


class Fit:
    """K-fold selection over candidates from statistics. fit(train_stats) -> [(A, c)] per candidate."""

    def __init__(self, fit, folds, tot, crit="r"):
        self.fit, self.tot = fit, tot
        self.oof = None
        for tr, te in folds:
            s = np.stack([pred_sums(A, c, te) for A, c in fit(tr)])
            self.oof = s if self.oof is None else self.oof + s
        if crit == "r":
            score = [np.nanmean(r_of(o)) for o in self.oof]
            self.best = int(np.nanargmax(score))
        else:
            self.best = int(np.argmin([np.nansum(sse_of(o)) for o in self.oof]))
        self.A, self.c = fit(tot)[self.best]
        self.fold_models = None

    def fold_model(self, folds):
        if self.fold_models is None:
            self.fold_models = [self.fit(tr)[self.best] for tr, _ in folds]
        return self.fold_models


def calibrated(A, c, oof, ym):
    """Per-contact least-squares gain on the pooled out-of-fold predictions (r unchanged, R^2 honest)."""
    n, sp, spp, spy, sy, _ = oof
    num = spy - ym * sp - ym * sy + n * ym ** 2
    den = spp - 2 * ym * sp + n * ym ** 2
    g = np.clip(num / np.maximum(den, 1e-12), 0, None)
    return A * g, ym + g * (c - ym)


def score(A, c, oof, tot, te, te_sessions=None):
    ym = tot.sy / tot.n
    s = pred_sums(A, c, te)
    Ac, cc = calibrated(A, c, oof, ym)
    sc = pred_sums(Ac, cc, te)
    r2 = 1 - sse_of(sc) / np.maximum(sc[5] - sc[4] ** 2 / sc[0], 1e-12)
    out = {"r": float(np.nanmean(r_of(s))), "r2": float(np.nanmean(r2)), "contacts": [float(x) for x in r_of(s)],
           "cv_contacts": [float(x) for x in r_of(oof)]}          # K-fold r on calibration (reliability map)
    if te_sessions:                                            # calibration longevity: r per test session
        out["sessions"] = {k: float(np.nanmean(r_of(pred_sums(A, c, v)))) for k, v in te_sessions.items()}
    return out


def embed(A, c, idx, D):
    out = np.zeros((D, A.shape[1]))
    out[idx] = A
    return out, c


# ----------------------------------------------------------------------------- features
def load_encoder(path, dev):
    ck = torch.load(path, map_location=dev, weights_only=False)
    enc = TemporalEncoder(**ck["cfg"]).to(dev)
    enc.load_state_dict(ck["state"])
    return enc.eval()


def warped(enc, s, dev):
    """Same encoder with every kernel time-scaled by s: h_s(t) = h(s t), i.e. frequencies x 1/s."""
    out = TemporalEncoder(**enc.cfg).to(dev)
    w = enc.conv.weight.detach().cpu().numpy()[:, 0]
    H = w.shape[1] // 2
    t = np.arange(-H, H + 1)
    ws = np.stack([np.interp(t / s, t, k, left=0.0, right=0.0) for k in w])
    ws *= np.linalg.norm(w, axis=1, keepdims=True) / (np.linalg.norm(ws, axis=1, keepdims=True) + 1e-12)
    with torch.no_grad():
        out.conv.weight.copy_(torch.tensor(ws, dtype=torch.float32)[:, None])
    return out.eval()


def high_mask(enc, use_init):
    """Per-feature mask (length n_feat): True for quadrature pairs >= HIGH_HZ and their envelopes."""
    cfg = enc.cfg
    P = cfg["n_pairs"]
    if use_init:
        f = np.geomspace(cfg["fmin"], cfg["fmax"], P)
    else:
        w = enc.conv.weight.detach().cpu().numpy()[:, 0]
        spec = np.abs(np.fft.rfft(w, 4096, axis=1)) ** 2
        fr = np.fft.rfftfreq(4096, 1 / FS)
        f = fr[np.argmax(spec[0:2 * P:2] + spec[1:2 * P:2], 1)]
    m = np.zeros(enc.n_feat, bool)
    for p in np.where(f >= HIGH_HZ)[0]:
        m[[2 * p, 2 * p + 1]] = True
        if cfg["envelope"]:
            m[2 * P + cfg["n_free"] + p] = True
    return m, f


def band_targets(y):
    lp = [sosfiltfilt(_LP[c], y, axis=1) for c in Y_CUTS]
    parts = [lp[0]] + [lp[i] - lp[i - 1] for i in range(1, len(lp))] + [y - lp[-1]]
    return np.stack(parts).astype(np.float32)


# ----------------------------------------------------------------------------- per subject
def load_subject(path):
    d = torch.load(path, weights_only=False)
    eeg, tgt = d["eeg_car"].half().float().numpy(), d["ieeg_bip"].half().float().numpy()
    b = d["trial_bounds"].numpy()
    ses = list(d.get("trial_session", ["ses-01"] * len(b)))
    return [eeg[:, s0:s1] for s0, s1 in b], [tgt[:, s0:s1] for s0, s1 in b], ses


def _robust(x):
    med = np.median(x, axis=1, keepdims=True)
    mad = np.median(np.abs(x - med), axis=1, keepdims=True) * 1.4826
    return med, np.where(mad < 1e-12, 1.0, mad)


def strict_renorm(E, Y, ses, stats, n_cal):
    """
    Undo the whole-session robust normalisation of preprocessing (x * MAD_s + median_s) and re-normalise every
    trial with the median / MAD of the CALIBRATION trials only (first n_cal), for EEG and target alike:
    no statistic of a test trial is used anywhere (strict, deployment-faithful normalisation).
    """
    def redo(X, key):
        raw = []
        for x, s in zip(X, ses):
            st = stats["norm_stats"][key][s]
            raw.append(x * np.asarray(st["mad"], np.float32)[:, None] + np.asarray(st["med"], np.float32)[:, None])
        med, mad = _robust(np.concatenate(raw[:n_cal], 1))
        return [((r - med) / mad).astype(np.float32) for r in raw]
    return redo(E, "eeg_car"), redo(Y, "ieeg_bip")


def analyze(E, Y, ses, enc, seed_encs, a, dev, log):
    amp = dev.type == "cuda"
    zoo = not a.only_combo
    n, C, K = len(E), E[0].shape[0], Y[0].shape[0]
    _, test = chrono_split(n, 0.2, gap=1)
    enc0 = TemporalEncoder(**enc.cfg).to(dev).eval()
    F = enc.n_feat
    hi_enc, f_enc = high_mask(enc, use_init=False)
    hi_e0, _ = high_mask(enc0, use_init=True)
    f1, f0 = np_feat(enc, dev, amp), np_feat(enc0, dev, amp)
    fss = [np_feat(se, dev, amp) for se in seed_encs]
    fs1 = fss[0] if fss else None
    fw = {s: np_feat(warped(enc, s, dev), dev, amp) for s in WARPS} if zoo else {}
    e0_rows = np.where(np.tile(hi_e0, C))[0]
    seeds = [f"seed{j}" for j in range(len(fss))]

    # main view = [enc | enc0 high-frequency rows (zoo only) | LAG9 | seed encoders]
    blocks = [("enc", C * F)] + ([("enc0hi", len(e0_rows))] if zoo else []) + [("lag9", C * len(DEFAULT_LAGS))]
    blocks += [(b, C * se.n_feat) for b, se in zip(seeds, seed_encs)]     # encoders may differ (e.g. no_env)
    idx, o = {}, 0
    for name, size in blocks:
        idx[name] = np.arange(o, o + size)
        o += size
    D = o
    hi_rows = np.where(np.tile(hi_enc, C))[0]
    lo_rows = np.where(~np.tile(hi_enc, C))[0]

    def main_x(e):
        x1 = f1(e)
        parts = [x1] + ([f0(e)[e0_rows]] if zoo else []) + [LAG9(e)] + [f(e) for f in fss]
        return np.concatenate(parts).astype(np.float32), x1

    # session statistics of every encoder's features (EEG of every trial, no target)
    enc_fns = {"enc": f1, **dict(zip(seeds, fss))}
    sess_sum = {}
    for i in range(n):
        for b, fn in enc_fns.items():
            x = fn(E[i]).astype(np.float64)
            s = sess_sum.setdefault((b, ses[i]), [0, np.zeros(len(idx[b])), np.zeros(len(idx[b]))])
            s[0] += x.shape[1]
            s[1] += x.sum(1)
            s[2] += (x * x).sum(1)
    sess_tf = {k: (v[1] / v[0], np.sqrt(np.clip(v[2] / v[0] - (v[1] / v[0]) ** 2, 0, None)) + 1e-6)
               for k, v in sess_sum.items()}

    def tf_for(names, session):
        """Per-session standardisation of encoder blocks; identity for other blocks (e.g. LAG9)."""
        m = [sess_tf[(b, session)][0] if (b, session) in sess_tf else np.zeros(len(idx[b])) for b in names]
        d = [sess_tf[(b, session)][1] if (b, session) in sess_tf else np.ones(len(idx[b])) for b in names]
        return np.concatenate(m), np.concatenate(d)

    # test statistics (per session, for the session-standardised readout)
    te_main, te_w = {}, {s: St(C * F, K) for s in fw}
    for i in test:
        X, _ = main_x(E[i])
        te_main.setdefault(ses[i], St(D, K)).add(X, Y[i])
        for s in fw:
            te_w[s].add(fw[s](E[i]).astype(np.float32), Y[i])
    te = total(list(te_main.values()))
    te_enc = te.take(idx["enc"])
    te_sess = total([v.take(idx["enc"], transform=tf_for(["enc"], k)) for k, v in te_main.items()])

    order = list(dict.fromkeys(ses))                     # sessions in recording order
    res = {"n_trials": n, "n_test": len(test), "n_sessions": len({k[1] for k in sess_sum}),
           "session_order": order, "test_trials_per_session": {k: sum(ses[i] == k for i in test) for k in order},
           "n_encoders": 1 + len(fss),
           "pair_peak_hz": [float(x) for x in f_enc], "fracs": {}}
    for frac in a.fracs:
        n_c = max(3, int(round(n * frac)))
        if n_c >= test[0]:
            continue
        t0 = time.time()
        fold_ids = _blocks(n_c, 5)
        cal = {}                                 # (fold, session) -> St (main view, with band targets)
        cal_w = {s: [St(C * F, K) for _ in fold_ids] for s in fw}
        nbt = len(Y_CUTS) + 1 if zoo else 0
        for fi, ids in enumerate(fold_ids):
            for i in ids:
                X, _ = main_x(E[i])
                cal.setdefault((fi, ses[i]), St(D, K, nb=nbt, Db=C * F if zoo else 0)).add(
                    X, Y[i], band_targets(Y[i]) if zoo else None)
                for s in fw:
                    cal_w[s][fi].add(fw[s](E[i]).astype(np.float32), Y[i])
        fold_st = [total([v for (f, _), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
        tot = total(fold_st)

        def folds_of(take):
            return [(take(tot - fs), take(fs)) for fs in fold_st]

        out, extra = {}, {}

        def run(name, take, fit, test_st, crit="r", note=None):
            fl = folds_of(take)
            m = Fit(fit, fl, take(tot), crit)
            out[name] = score(m.A, m.c, m.oof[m.best], take(tot), test_st,
                              {k: take(v) for k, v in te_main.items()})
            if note:
                extra[name] = note(m)
            return m, fl

        t_enc = lambda s: s.take(idx["enc"])                                    # noqa: E731
        m_enc, fl_enc = run("enc", t_enc, lambda tr: ridge_models(tr, ALPHAS_WIDE), te_enc,
                            note=lambda m: ALPHAS_WIDE[m.best])
        t_lag = lambda s: s.take(idx["lag9"])                                   # noqa: E731
        m_base, fl_base = run("base", t_lag, lambda tr: ridge_models(tr, ALPHAS), te.take(idx["lag9"]))
        store = {}                       # model -> (feature rows, calibrated A, c, session-standardised blocks)
        for nm, m_, rows in (("enc", m_enc, idx["enc"]), ("base", m_base, idx["lag9"])):
            ts = tot.take(rows)
            store[nm] = (rows, *calibrated(m_.A, m_.c, m_.oof[m_.best], ts.sy / ts.n), None)

        if zoo:
            lo = idx["enc"][lo_rows]
            run("enc_low", lambda s: s.take(lo), lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(lo))

            gmask = np.zeros(C * F, bool)
            gmask[hi_rows] = True
            cands = [(g, al) for g in GROUP_SCALES for al in ALPHAS_WIDE]

            def fit_group(tr):
                outm = []
                for g in GROUP_SCALES:
                    outm += ridge_models(tr, ALPHAS_WIDE, np.where(gmask, g, 1.0))
                return outm
            run("enc_group", t_enc, fit_group, te_enc, note=lambda m: cands[m.best][0])

            def added(name, block):
                sel = np.concatenate([idx["enc"], idx[block]])
                mask = np.r_[np.zeros(len(idx["enc"]), bool), np.ones(len(idx[block]), bool)]
                cands_ = [(g, al) for g in ADD_SCALES for al in ALPHAS_WIDE]

                def fit(tr):
                    outm = []
                    for g in ADD_SCALES:
                        outm += ridge_models(tr, ALPHAS_WIDE, np.where(mask, g, 1.0))
                    return outm
                run(name, lambda s: s.take(sel), fit, te.take(sel), note=lambda m: cands_[m.best][0])
            added("enc_enc0hi", "enc0hi")
            added("enc_lag9", "lag9")

            # one ridge per target band (penalty by MSE on that band), predictions summed
            nb = len(Y_CUTS) + 1
            bm = [Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), folds_of(lambda s, b=b: s.take(idx["enc"], band=b)),
                      tot.take(idx["enc"], band=b), crit="mse") for b in range(nb)]
            A = sum(m.A for m in bm)
            c = sum(m.c for m in bm)
            fms = [m.fold_model([(tr, None) for tr, _ in folds_of(lambda s, b=b: s.take(idx["enc"], band=b))])
                   for b, m in enumerate(bm)]
            oof = sum(pred_sums(sum(fms[b][fi][0] for b in range(nb)), sum(fms[b][fi][1] for b in range(nb)), te_f)
                      for fi, (_, te_f) in enumerate(fl_enc))
            out["enc_bands"] = score(A, c, oof, t_enc(tot), te_enc)
            extra["enc_bands"] = [ALPHAS_WIDE[m.best] for m in bm]

            # per-session standardisation of the encoder features
            sess_tot = total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()])
            sess_folds = [total([v.take(idx["enc"], transform=tf_for(["enc"], s)) for (f, s), v in cal.items()
                                 if f == fi])
                          for fi in range(len(fold_ids))]
            m = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(sess_tot - fs, fs) for fs in sess_folds], sess_tot)
            out["enc_sess"] = score(m.A, m.c, m.oof[m.best], sess_tot, te_sess)

            # individual frequency adaptation: pick s in {0.9, 1, 1.1} by K-fold r
            best = ("1.0", m_enc, np.nanmean(r_of(m_enc.oof[m_enc.best])), te_enc, t_enc(tot))
            for s in WARPS:
                tw = total(cal_w[s])
                mw = Fit(lambda tr: ridge_models(tr, ALPHAS_WIDE), [(tw - fs, fs) for fs in cal_w[s]], tw)
                sw = np.nanmean(r_of(mw.oof[mw.best]))
                if sw > best[2]:
                    best = (f"{s:g}", mw, sw, te_w[s], tw)
            out["enc_warp"] = score(best[1].A, best[1].c, best[1].oof[best[1].best], best[4], best[3])
            extra["enc_warp"] = best[0]

            # structured (bilinear) readout
            if not a.no_bilin:
                run("enc_bilin", t_enc, lambda tr: bilinear_models(tr, C, F, BILIN), te_enc,
                    note=lambda m: list(BILIN[m.best]))

        # ensembles: enc + base, enc + seed-1 (weights chosen by pooled out-of-fold r)
        def ensemble(name, m1, fl1, i1, m2, fl2, i2):
            sel = np.concatenate([i1, i2])
            n1 = len(i1)
            fm1 = m1.fold_model(fl1)
            fm2 = m2.fold_model(fl2)
            t_sel = lambda s: s.take(sel)                                       # noqa: E731
            fl = folds_of(t_sel)
            oofs = []
            for w in ENS_W:
                oofs.append(sum(pred_sums(np.r_[w * fm1[fi][0], (1 - w) * fm2[fi][0]], w * fm1[fi][1] +
                                          (1 - w) * fm2[fi][1], te_f) for fi, (_, te_f) in enumerate(fl)))
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            w = ENS_W[wi]
            A_ = np.r_[w * m1.A, (1 - w) * m2.A]
            out[name] = score(A_, w * m1.c + (1 - w) * m2.c, oofs[wi], t_sel(tot), te.take(sel))
            extra[name] = w
            assert len(A_) == len(sel) and n1 == len(m1.A)
        ensemble("ens_base", m_enc, fl_enc, idx["enc"], m_base, fl_base, idx["lag9"])
        if fss:
            t_seed = lambda s: s.take(idx["seed0"])                             # noqa: E731
            m_seed, fl_seed = run("seed", t_seed, lambda tr: ridge_models(tr, ALPHAS_WIDE), te.take(idx["seed0"]))
            ensemble("ens_seed", m_enc, fl_enc, idx["enc"], m_seed, fl_seed, idx["seed0"])

        # combinations of the three Phase-4 winners: several encoders averaged (ens_seed), session
        # standardisation (enc_sess) and mixing with LAG9 ridge (ens_base); exact from statistics
        def mixture(name, encs, sess, fixed_w=None):
            names = encs + ["lag9"]
            sel = np.concatenate([idx[b] for b in names])

            def T(st, session):
                return st.take(sel, transform=tf_for(names, session) if sess else None)
            folds_T = [total([T(v, ss) for (f, ss), v in cal.items() if f == fi]) for fi in range(len(fold_ids))]
            tot_T = total(folds_T)
            te_T = total([T(v, ss) for ss, v in te_main.items()])
            comps, o_ = [], 0
            for b in names:
                loc = np.arange(o_, o_ + len(idx[b]))
                o_ += len(idx[b])
                al = ALPHAS if b == "lag9" else ALPHAS_WIDE
                fl = [((tot_T - fs).take(loc), fs.take(loc)) for fs in folds_T]
                m = Fit(lambda tr, al=al: ridge_models(tr, al), fl, tot_T.take(loc))
                comps.append((b, loc, m, m.fold_model(fl)))

            def combine(w, models):
                A_, c_ = np.zeros((len(sel), K)), np.zeros(K)
                for (b, loc, _, _), (Ab, cb) in zip(comps, models):
                    wt = (1 - w) if b == "lag9" else w / len(encs)
                    A_[loc] += wt * Ab
                    c_ += wt * cb
                return A_, c_
            ws = ENS_W if fixed_w is None else (fixed_w,)
            oofs = [sum(pred_sums(*combine(w, [cm[3][fi] for cm in comps]), fs) for fi, fs in enumerate(folds_T))
                    for w in ws]
            wi = int(np.nanargmax([np.nanmean(r_of(o)) for o in oofs]))
            A_, c_ = combine(ws[wi], [(cm[2].A, cm[2].c) for cm in comps])
            out[name] = score(A_, c_, oofs[wi], tot_T, te_T, {ss: T(v, ss) for ss, v in te_main.items()})
            store[name] = (sel, *calibrated(A_, c_, oofs[wi], tot_T.sy / tot_T.n), names if sess else None)
            extra[name] = ws[wi]

        all_encs = ["enc"] + seeds
        if not zoo and not a.strict:
            mixture("enc_sess", ["enc"], sess=True, fixed_w=1.0)
        if seeds:
            mixture("ens_all", all_encs, sess=False, fixed_w=1.0)
            if not a.strict:
                mixture("ens_all_sess", all_encs, sess=True, fixed_w=1.0)
        if not a.strict:                         # session standardisation uses the EEG of every trial (transductive)
            mixture("combo", all_encs, sess=True)
        mixture("combo_nosess", all_encs, sess=False)

        if a.explicit and abs(frac - a.explicit_frac) < 1e-9:
            # explicit test predictions of the final models: band / 50 Hz / single-trial metrics
            res["explicit"] = {}
            Yt = [Y[i] for i in test]
            for nm in [m for m in ("base", "enc", "combo", "combo_nosess") if m in store]:
                rows, A_, c_, names = store[nm]
                P = []
                for i in test:
                    X = main_x(E[i])[0][rows].astype(np.float64)
                    if names:
                        mm, dd = tf_for(names, ses[i])
                        X = (X - mm[:, None]) / dd[:, None]
                    P.append((X.T @ A_ + c_).T.astype(np.float32))
                met = per_contact_metrics(P, Yt, FS, bands=True)
                met["r_no50"], met["r_induced"] = r_without_line(P, Yt), r_single_trial(P, Yt)
                res["explicit"][nm] = {"frac": frac, "mean": {k: float(np.nanmean(v)) for k, v in met.items()},
                                       "r": [float(x) for x in met["r"]],
                                       "r_induced": [float(x) for x in met["r_induced"]]}
            log("    explicit: " + " ".join(f"{k} r={v['mean']['r']:.4f} (stats {out[k]['r']:.4f}) "
                                          f"r_ind={v['mean']['r_induced']:.4f}"
                                          for k, v in res["explicit"].items()))
        mins = sum(E[i].shape[1] for i in range(n_c)) / FS / 60
        res["fracs"][f"{frac:g}"] = {"n_calib": n_c, "minutes": mins, "models": out, "choice": extra,
                                     "calib_sessions": sorted({ses[i] for i in range(n_c)}, key=order.index)}
        log(f"    f={frac:g} ({mins:.1f} min, {time.time() - t0:.0f}s) " +
            " ".join(f"{k}={v['r']:.3f}" for k, v in out.items()))
    return res


# ----------------------------------------------------------------------------- summary
def summarize(results, out_dir, log):
    from scipy.stats import wilcoxon
    lines, summ = [], {}
    fracs = sorted({f for r in results for f in r["fracs"]}, key=float)
    for f in fracs:
        rows = [r["fracs"][f] for r in results if f in r["fracs"]]
        models = [m for m in rows[0]["models"] if all(m in r["models"] for r in rows)]
        M = {m: np.array([r["models"][m]["r"] for r in rows]) for m in models}
        R2 = {m: np.array([r["models"][m]["r2"] for r in rows]) for m in models}
        mins = np.mean([r["minutes"] for r in rows])
        lines += [f"\n### calib fraction {f}  ({len(rows)} subjects, mean {mins:.1f} min)",
                  "| model | mean r | vs enc | wins vs enc | p vs enc | vs base | R² (calibrated) | K-fold choices |",
                  "|---|---|---|---|---|---|---|---|"]
        summ[f] = {}
        for m in models:
            d = M[m] - M["enc"]
            try:
                p = float(wilcoxon(M[m], M["enc"]).pvalue) if m != "enc" and np.any(d != 0) else float("nan")
            except ValueError:
                p = float("nan")
            ch = [r["choice"].get(m) for r in rows if m in r["choice"]]
            ch = [tuple(x) if isinstance(x, list) else x for x in ch]
            if ch and m != "enc_bands":
                vals, cnt = np.unique([str(x) for x in ch], return_counts=True)
                chs = ", ".join(f"{v}×{c}" for v, c in zip(vals, cnt))
            else:
                chs = ""
            summ[f][m] = {"mean": float(M[m].mean()), "diff_enc": float(d.mean()), "wins_enc": int((d > 0).sum()),
                          "p_enc": p, "diff_base": float((M[m] - M["base"]).mean()), "r2": float(R2[m].mean())}
            lines.append(f"| {m} | {M[m].mean():.4f} | {d.mean():+.4f} | {(d > 0).sum()}/{len(d)} | {p:.4f} | "
                         f"{(M[m] - M['base']).mean():+.4f} | {R2[m].mean():.3f} | {chs} |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "readout.md"), "w", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "readout.json"), "w") as fh:
        json.dump(summ, fh, indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--run", required=True, help="output dir of s2i.shared (enc_<sub>.pt)")
    ap.add_argument("--seed_run", default=None, help="second run for the seed ensemble (enc_<sub>.pt)")
    ap.add_argument("--seed_runs", nargs="*", default=[], help="further runs (encoders) for the ensemble")
    ap.add_argument("--explicit", action="store_true",
                    help="also predict the test trials explicitly with base / enc / combo (all metrics)")
    ap.add_argument("--explicit_frac", type=float, default=0.2)
    ap.add_argument("--strict", action="store_true",
                    help="re-normalise EEG and target with calibration-trial statistics only (needs <sub>_normstats.pt "
                         "from preprocess --stats_only); no session standardisation; requires a single --fracs value")
    ap.add_argument("--stats_dir", default=None, help="folder with <sub>_normstats.pt (default: --data)")
    ap.add_argument("--untrained", action="store_true",
                    help="replace every encoder by its Morlet initialisation (fixed filter bank, no meta-learning)")
    ap.add_argument("--only_combo", action="store_true",
                    help="skip the Phase-4 variants already rejected; compute enc, base, ensembles and combos")
    ap.add_argument("--out_name", default=None, help="output sub-folder (default: readout / readout_combo)")
    ap.add_argument("--fracs", type=float, nargs="*", default=[0.05, 0.1, 0.2])
    ap.add_argument("--subjects", nargs="*", default=None)
    ap.add_argument("--no_bilin", action="store_true")
    ap.add_argument("--overwrite", action="store_true")
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    if a.strict:
        assert len(a.fracs) == 1, "--strict needs exactly one calibration fraction (the statistics depend on it)"
    out = os.path.join(a.run, a.out_name or ("readout_strict" if a.strict else
                                             "readout_combo" if a.only_combo else "readout"))
    os.makedirs(out, exist_ok=True)
    log_f = open(os.path.join(out, "readout.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    torch.set_grad_enabled(False)
    global _DEV
    _DEV = dev
    seed_runs = ([a.seed_run] if a.seed_run else []) + list(a.seed_runs)
    log(f"config: {vars(a)}")
    results = []
    for ep in sorted(glob.glob(os.path.join(a.run, "enc_sub-*.pt"))):
        sid = os.path.basename(ep)[4:-3]
        if a.subjects and sid not in a.subjects:
            continue
        fp = os.path.join(out, f"readout_{sid}.json")
        sps = [sp for sp in (os.path.join(r_, f"enc_{sid}.pt") for r_ in seed_runs) if os.path.exists(sp)]
        if os.path.exists(fp) and not a.overwrite:
            done = json.load(open(fp))
            # resume: keep a finished subject only if it used the same number of encoders as now
            if done.get("n_encoders", 1 + len(sps)) == 1 + len(sps):
                results.append(done)
                log(f"[{sid}] already done, loaded")
                continue
            log(f"[{sid}] done with {done['n_encoders']} encoders, now {1 + len(sps)}: recomputing")
        t0 = time.time()
        enc = load_encoder(ep, dev)
        seed_encs = [load_encoder(sp, dev) for sp in sps]
        if a.untrained:
            enc = TemporalEncoder(**enc.cfg).to(dev).eval()
            seed_encs = [TemporalEncoder(**se.cfg).to(dev).eval() for se in seed_encs]
        E, Y, ses = load_subject(os.path.join(a.data, f"{sid}.pt"))
        if a.strict:
            st = torch.load(os.path.join(a.stats_dir or a.data, f"{sid}_normstats.pt"), weights_only=False)
            assert len(st["trial_bounds"]) == len(E) and list(st["trial_session"]) == ses, "stats / data mismatch"
            E, Y = strict_renorm(E, Y, ses, st, max(3, int(round(len(E) * a.fracs[0]))))
        log(f"[{sid}] {len(E)} trials, {E[0].shape[0]} scalp, {Y[0].shape[0]} targets, "
            f"{len(set(ses))} sessions, extra encoders: {len(seed_encs)}")
        r = analyze(E, Y, ses, enc, seed_encs, a, dev, log)
        r["subject"] = sid
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results.append(r)
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
    if results:
        summarize(results, out, log)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/zeroshot.py
"""
Phase 6 (CPU) - calibration-free (zero-shot) prediction of intracranial BAND POWER.

Why: waveforms do not transfer across patients (Phase 1a): a bipolar channel flips sign with the
shaft orientation, which differs between patients. Band power is sign-invariant, and region-level
theta power transferred across patients in Subramanian et al. 2025 (hd-EEG, R^2 0.10-0.15).
Question: with the sparse 10-20 montages of ds004752, can the power time course of a contact in a NEW
patient be predicted with NO intracranial data from that patient?

Signals: 1-s non-overlapping windows inside each trial; log band power (delta 1-4, theta 4-8,
alpha 8-13, beta 13-30 Hz) of every bipolar contact (target) and every scalp channel (input,
z-scored per session using EEG only).

Models (leave-one-subject-out; test windows = trials after the 20 % split, as in all phases)
  zs_task     task timing only: mean z-power at the same window position in the trial, from the other
              patients (no EEG) -> how much of the power course is simply stimulus-locked
  zs_global   mean scalp power of the same band (no training)
  zs_nearest  power of the scalp channel nearest to the contact (no training)
  zs_geo      learned geometry readout: scalp power of every band averaged with Gaussian distance
              kernels (30..140 mm) around the contact + the global mean; one linear model per target
              band fitted on the OTHER patients (penalty by leave-one-patient-out inside them),
              applied unchanged to the new patient
  zs_geo_shuf CONTROL (anatomy mismatch): zs_geo with the test patient's contact positions permuted
              among its contacts -> if it equals zs_geo, the geometry contributes nothing
  cal_ridge   reference WITH calibration: ridge per contact on all scalp channels x bands, fitted on the
              first 20 % of the patient's own trials (penalty by 5-fold r)
  zs_cal      hybrid: per contact, ridge on [zs_geo prediction, all scalp channels x bands] fitted on the same
              calibration windows; the zs_geo term and the intercept are NOT penalised, so a large penalty
              falls back to (rescaled) zs_geo and calibration only adds a shrunk correction
Metrics per contact on the test windows: r, and r_induced (each side minus its own mean at the same
window position across the test trials, i.e. beyond the task-locked part).

Note: scalp positions are the MNE standard 10-20 template (head frame ~ fsaverage/MNI305), contacts are
in MNI; the frames agree only approximately, which the distance kernels partly absorb.

Usage
-----
python -m s2i.zeroshot --data DATA_DIR --out RUNS_DIR/zeroshot_power
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import torch
from scipy.signal import butter, sosfiltfilt

FS, WIN, EDGE = 200, 200, 200          # 1-s windows; 1 s dropped at each trial edge
PBANDS = (("delta", 1, 4), ("theta", 4, 8), ("alpha", 8, 13), ("beta", 13, 30))
_PS = [butter(4, (lo, hi), btype="band", fs=FS, output="sos") for _, lo, hi in PBANDS]
CENTERS, WIDTH = np.array([30.0, 50.0, 70.0, 90.0, 110.0, 140.0]), 15.0
ALPHAS_ZS = (1e-2, 1e-1, 1.0, 10.0, 100.0, 1e3)
MODELS = ("zs_task", "zs_global", "zs_nearest", "zs_geo", "zs_geo_shuf", "cal_ridge", "zs_cal")


# ----------------------------------------------------------------------------- data
def logpow(x):
    """
    (channels, L) one trial -> (n_windows, channels, n_bands) log mean power per 1-s window.
    The first and last EDGE samples are dropped: per-trial band-pass filtering rings at the trial edges
    in BOTH modalities, which creates spurious, position-locked power correlation (synthetic check:
    delta r ~0.5 with no coupling at all).
    """
    nw = (x.shape[1] - 2 * EDGE) // WIN
    if nw <= 0:
        return np.zeros((0, x.shape[0], len(_PS)))
    out = []
    for sos in _PS:
        b = sosfiltfilt(sos, x, axis=1)[:, EDGE:EDGE + nw * WIN]
        out.append(np.log((b.reshape(x.shape[0], nw, WIN) ** 2).mean(2) + 1e-6))
    return np.stack(out, -1).transpose(1, 0, 2)


def load(path):
    d = torch.load(path, weights_only=False)
    eeg, tgt = d["eeg_car"].half().float().numpy(), d["ieeg_bip"].half().float().numpy()
    bounds = d["trial_bounds"].numpy()
    ses = list(d.get("trial_session", ["ses-01"] * len(bounds)))
    X, Y, tr, pos, ss = [], [], [], [], []
    for i, (s0, s1) in enumerate(bounds):
        x, y = logpow(eeg[:, s0:s1]), logpow(tgt[:, s0:s1])
        X.append(x)
        Y.append(y)
        tr += [i] * len(x)
        pos += list(range(len(x)))
        ss += [ses[i]] * len(x)
    X, Y, ss = np.concatenate(X), np.concatenate(Y), np.array(ss)
    for s in np.unique(ss):                                   # scalp: z-score per session (EEG only)
        m = ss == s
        X[m] = (X[m] - X[m].mean(0)) / (X[m].std(0) + 1e-6)
    return {"sid": d["subject"], "X": X.astype(np.float32), "Y": Y.astype(np.float32), "trial": np.array(tr),
            "pos": np.array(pos), "n_trials": len(bounds), "eeg_pos": np.asarray(d["eeg_pos"], float),
            "con_pos": np.asarray(d["bip_mni"], float), "depth": list(d["bip_depth"]),
            "region": list(d["bip_region"])}


def zscore(Y):
    return (Y - Y.mean(0)) / (Y.std(0) + 1e-6)


def geo_features(S):
    """(n_windows, K, (J + 1) * n_bands): scalp power averaged with distance kernels around each contact."""
    d = np.linalg.norm(S["con_pos"][:, None] - S["eeg_pos"][None], axis=2)          # (K, C)
    phi = np.exp(-(d[:, None, :] - CENTERS[None, :, None]) ** 2 / (2 * WIDTH ** 2))   # (K, J, C)
    phi = np.concatenate([phi, np.ones((len(d), 1, d.shape[1]))], 1)                 # + global mean
    tot = phi.sum(2, keepdims=True)
    phi = np.where(tot > 1e-3, phi / np.maximum(tot, 1e-12), 0.0)                    # empty ring -> 0
    F = np.einsum("kjc,wcb->wkjb", phi, S["X"])
    return F.reshape(F.shape[0], F.shape[1], -1).astype(np.float32), d


# ----------------------------------------------------------------------------- metrics
def r_cols(P, T):
    P, T = P - P.mean(0), T - T.mean(0)
    den = np.sqrt((P * P).sum(0) * (T * T).sum(0))
    return np.where(den > 0, (P * T).sum(0) / np.maximum(den, 1e-12), np.nan)


def remove_position_mean(Z, pos):
    out = Z.copy()
    for p in np.unique(pos):
        m = pos == p
        out[m] -= Z[m].mean(0)
    return out


def ridge_fit(A, B, alpha, free=None):
    """Ridge with penalty alpha * mean diagonal; `free` = indices of unpenalised columns."""
    D = len(A)
    pen = np.ones(D)
    if free is not None:
        pen[free] = 0.0
    return np.linalg.solve(A + alpha * np.trace(A) / D * np.diag(pen) + 1e-9 * np.eye(D), B)


# ----------------------------------------------------------------------------- LOSO
def run_fold(t, subs, feats, log):
    S = subs[t]
    n = S["n_trials"]
    n_c = max(3, int(round(n * 0.2)))
    cal, test = S["trial"] < n_c, S["trial"] >= n_c + 1
    others = [s for s in range(len(subs)) if s != t]
    nB = len(PBANDS)
    Yt = S["Y"][test]                                                 # (w, K, nB)
    res = {"subject": S["sid"], "n_test_windows": int(test.sum()), "bands": {}}

    # zero-shot geometry readout: normal equations per training patient and band
    stats = {}
    for s in others:
        F = feats[s][0]
        Fi = np.concatenate([F, np.ones(F.shape[:2] + (1,), np.float32)], 2)         # + intercept
        Yz = zscore(subs[s]["Y"])
        A = np.einsum("wkd,wke->de", Fi, Fi)
        B = np.einsum("wkd,wkb->db", Fi, Yz)
        stats[s] = (A, B, Fi, Yz)
    # penalty by leave-one-patient-out among the training patients (mean r over contacts)
    A_all, B_all = sum(stats[s][0] for s in others), sum(stats[s][1] for s in others)
    score = np.zeros((len(ALPHAS_ZS), nB))
    for s in others:
        A, B, Fi, Yz = stats[s]
        for ai, al in enumerate(ALPHAS_ZS):
            th = ridge_fit(A_all - A, B_all - B, al)
            P = np.einsum("wkd,db->wkb", Fi, th)
            for b in range(nB):
                score[ai, b] += np.nanmean(r_cols(P[:, :, b], Yz[:, :, b]))
    best = score.argmax(0)
    Ft = feats[t][0][test]
    Fti = np.concatenate([Ft, np.ones(Ft.shape[:2] + (1,), np.float32)], 2)

    # task timing template from the other patients (z-power by window position)
    tmpl = {}
    for s in others:
        Yz, pos = zscore(subs[s]["Y"]).mean(1), subs[s]["pos"]                     # (w, nB)
        for p in np.unique(pos):
            tmpl.setdefault(p, []).append(Yz[pos == p].mean(0))
    tmpl = {p: np.mean(v, 0) for p, v in tmpl.items()}
    pos_t = S["pos"][test]

    # calibrated reference: ridge on all scalp channels x bands, the patient's first 20 % of trials
    Xc = S["X"][cal].reshape(cal.sum(), -1)
    Xt = S["X"][test].reshape(test.sum(), -1)
    Yc = S["Y"][cal]
    blocks = np.array_split(np.arange(n_c), 5)
    tr_c = S["trial"][cal]
    d_near = feats[t][1].argmin(1)
    perm = np.random.default_rng(0).permutation(len(S["con_pos"]))
    F_sh = geo_features(dict(S, con_pos=S["con_pos"][perm]))[0][test]
    F_sh = np.concatenate([F_sh, np.ones(F_sh.shape[:2] + (1,), np.float32)], 2)
    Fci = np.concatenate([feats[t][0][cal], np.ones((cal.sum(), feats[t][0].shape[1], 1), np.float32)], 2)

    for b, (bname, _, _) in enumerate(PBANDS):
        th = ridge_fit(A_all, B_all, ALPHAS_ZS[best[b]])[:, b]
        preds = {
            "zs_task": np.repeat(np.stack([tmpl.get(p, np.zeros(nB))[b] for p in pos_t])[:, None], Yt.shape[1], 1),
            "zs_global": np.repeat(S["X"][test][:, :, b].mean(1, keepdims=True), Yt.shape[1], 1),
            "zs_nearest": S["X"][test][:, d_near, b],
            "zs_geo": Fti @ th,
            "zs_geo_shuf": F_sh @ th,
        }
        # calibrated ridge (shared penalty by 5-fold r over trial blocks)
        sc = []
        for al in ALPHAS_ZS:
            rr = []
            for blk in blocks:
                ho = np.isin(tr_c, blk)
                Xa = np.c_[Xc[~ho], np.ones((~ho).sum())]
                W = ridge_fit(Xa.T @ Xa, Xa.T @ Yc[~ho][:, :, b], al)
                rr.append(r_cols(np.c_[Xc[ho], np.ones(ho.sum())] @ W, Yc[ho][:, :, b]))
            sc.append(np.nanmean(rr))
        Xa = np.c_[Xc, np.ones(len(Xc))]
        W = ridge_fit(Xa.T @ Xa, Xa.T @ Yc[:, :, b], ALPHAS_ZS[int(np.nanargmax(sc))])
        preds["cal_ridge"] = np.c_[Xt, np.ones(len(Xt))] @ W

        # hybrid: zero-shot prediction as an extra (standardised) feature of a per-contact calibrated ridge
        zc, zt = Fci @ th, preds["zs_geo"]                                  # (w_cal, K), (w_test, K)
        mu_, sd_ = zc.mean(0), zc.std(0) + 1e-6
        zc, zt = (zc - mu_) / sd_, (zt - mu_) / sd_
        sc = np.zeros(len(ALPHAS_ZS))
        for ai, al in enumerate(ALPHAS_ZS):
            rr = []
            for blk in blocks:
                ho = np.isin(tr_c, blk)
                P = np.zeros((ho.sum(), zc.shape[1]))
                for k in range(zc.shape[1]):
                    Xa = np.c_[zc[~ho, k], Xc[~ho], np.ones((~ho).sum())]
                    w = ridge_fit(Xa.T @ Xa, Xa.T @ Yc[~ho][:, k, b], al, free=[0, Xa.shape[1] - 1])
                    P[:, k] = np.c_[zc[ho, k], Xc[ho], np.ones(ho.sum())] @ w
                rr.append(r_cols(P, Yc[ho][:, :, b]))
            sc[ai] = np.nanmean(rr)
        al = ALPHAS_ZS[int(np.nanargmax(sc))]
        P = np.zeros_like(zt)
        for k in range(zc.shape[1]):
            Xa = np.c_[zc[:, k], Xc, np.ones(len(Xc))]
            w = ridge_fit(Xa.T @ Xa, Xa.T @ Yc[:, k, b], al, free=[0, Xa.shape[1] - 1])
            P[:, k] = np.c_[zt[:, k], Xt, np.ones(len(Xt))] @ w
        preds["zs_cal"] = P

        T = Yt[:, :, b]
        Ti = remove_position_mean(T, pos_t)
        out = {}
        for m, P in preds.items():
            r = r_cols(P, T)
            ri = r_cols(remove_position_mean(P, pos_t), Ti)
            out[m] = {"r": float(np.nanmean(r)), "r_induced": float(np.nanmean(ri)),
                      "contacts": [float(x) for x in r], "contacts_induced": [float(x) for x in ri]}
        out["zs_geo"]["alpha"] = ALPHAS_ZS[best[b]]
        out["zs_geo"]["weights"] = [float(x) for x in th]
        res["bands"][bname] = out
        log(f"    {bname:5s} " + " ".join(f"{m}={v['r']:+.3f}/{v['r_induced']:+.3f}" for m, v in out.items()))
    res["depth"], res["region"] = S["depth"], S["region"]
    return res


def summarize(results, out_dir, log):
    from scipy.stats import wilcoxon

    def wp(x, y=None):
        try:
            return float(wilcoxon(x, y).pvalue if y is not None else wilcoxon(x).pvalue)
        except ValueError:
            return float("nan")
    lines, summ = [], {}
    for metric in ("r", "r_induced"):
        lines += [f"\n### metric: {metric}  (mean over contacts, then over {len(results)} patients; "
                  "p = Wilcoxon over patients)",
                  "| band | " + " | ".join(MODELS) + " | zs_geo vs 0: p | zs_geo − zs_nearest: wins, p | "
                  "zs_geo − zs_task: wins, p | zs_geo − zs_geo_shuf: wins, p | zs_cal − cal_ridge: wins, p |",
                  "|---" * (len(MODELS) + 6) + "|"]
        for bname, _, _ in PBANDS:
            M = {m: np.array([r["bands"][bname][m][metric] for r in results]) for m in MODELS}
            g, nn, tk = M["zs_geo"], M["zs_nearest"], M["zs_task"]
            fin = np.isfinite(tk)
            summ.setdefault(metric, {})[bname] = {m: float(np.nanmean(v)) for m, v in M.items()}
            lines.append(f"| {bname} | " + " | ".join(f"{np.nanmean(M[m]):+.3f}" for m in MODELS) +
                         f" | {wp(g):.4f} | {(g > nn).sum()}/{len(g)}, {wp(g, nn):.4f} | " +
                         (f"{(g[fin] > tk[fin]).sum()}/{fin.sum()}, {wp(g[fin], tk[fin]):.4f} |" if fin.sum() >= 5
                          else "- |") +
                         f" {(g > M['zs_geo_shuf']).sum()}/{len(g)}, {wp(g, M['zs_geo_shuf']):.4f} |"
                         f" {(M['zs_cal'] > M['cal_ridge']).sum()}/{len(g)}, {wp(M['zs_cal'], M['cal_ridge']):.4f} |")
    # by depth (pooled contacts) for zs_geo and cal_ridge
    lines += ["\n### zs_geo and cal_ridge by depth (pooled contacts, r / r_induced)",
              "| band | depth | n | zs_geo r | zs_geo r_ind | cal_ridge r | cal_ridge r_ind |", "|---|---|---|---|---|---|---|"]
    for bname, _, _ in PBANDS:
        pools = {}
        for r in results:
            B = r["bands"][bname]
            for k, dep in enumerate(r["depth"]):
                pools.setdefault(dep, []).append([B["zs_geo"]["contacts"][k], B["zs_geo"]["contacts_induced"][k],
                                                  B["cal_ridge"]["contacts"][k], B["cal_ridge"]["contacts_induced"][k]])
        for dep, v in sorted(pools.items()):
            v = np.array(v, float)
            lines.append(f"| {bname} | {dep} | {len(v)} | " + " | ".join(f"{x:+.3f}" for x in np.nanmean(v, 0)) + " |")
    md = "\n".join(lines)
    log(md)
    with open(os.path.join(out_dir, "zeroshot.md"), "w", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "zeroshot.json"), "w") as fh:
        json.dump(summ, fh, indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--min_trials", type=int, default=20)
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "zeroshot.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    log(f"config: {vars(a)}")
    subs = []
    for p in sorted(glob.glob(os.path.join(a.data, "sub-[0-9][0-9].pt"))):
        S = load(p)
        if S["n_trials"] < a.min_trials or S["Y"].shape[1] == 0:
            continue
        subs.append(S)
        log(f"{S['sid']}: {len(S['X'])} windows, {S['X'].shape[1]} scalp, {S['Y'].shape[1]} contacts")
    feats = [geo_features(S) for S in subs]
    results = []
    for t in range(len(subs)):
        log(f"[{subs[t]['sid']}] test (r / r_induced)")
        results.append(run_fold(t, subs, feats, log))
    with open(os.path.join(a.out, "zeroshot_subjects.json"), "w") as fh:
        json.dump(results, fh)
    summarize(results, a.out, log)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE_DIR}/s2i/protocol.py
"""
Phase 5B (CPU) - protocol analysis: how much do the evaluation choices used in the literature inflate r?

Same data, same two models - base (LAG9 ridge) and enc (shared encoder trained WITHOUT the test patient +
ridge readout), penalties always chosen by 5-fold r inside the training data - only the protocol changes:

  clean            ours: first 20 % of trials (chronological) -> calibration, trials after a 1-trial gap -> test;
                   bipolar iEEG, CAR EEG
  rand_trials20    20 % of the trials drawn at random for calibration (interleaved in time with the test trials)
  rand_trials90    90 % random trials for training, 10 % for test (NeuroFlowNet-style amount and split)
  rand_seg80       every trial cut into 1-s segments, random 80/20 split of segments: test samples are temporally
                   adjacent to training samples (as with CAST's 50 %-overlapping 2-s windows split at random)
  mono_orig        clean split, but MONOPOLAR iEEG target and original-reference EEG (shared recording reference)
  mono_orig_seg80  monopolar + original reference + random 1-s segments (CAST-like data protocol)
Metrics (on the test samples)
  r_concat    ours: per-contact r over the concatenated test samples, mean over contacts
  r_window    per-contact r inside each 1-s test window, averaged
  r_nfn       NeuroFlowNet-style: per 1-s window the prediction is z-scored per channel, de-normalised with the
              TARGET window's own mean/std, and r is computed on the channel-flattened window; mean over windows
  r_nfn_null  the same metric for a random prediction (Gaussian noise): what the metric gives for free
Selection
  CAST-style "viable" patients: calibration K-fold r > 0.2; the mean is then reported on those patients only

Usage
-----
python -m s2i.protocol --data DATA_DIR --run RUNS_DIR/shared_bip_car --out RUNS_DIR/protocol
"""
import argparse
import glob
import json
import os
import sys
import time

import numpy as np
import torch

from s2i.baselines import DEFAULT_LAGS
from s2i.ceiling import ALPHAS, feat_lag
from s2i.metrics import per_contact_metrics
from s2i.readout import Fit, St, load_encoder, r_of, ridge_models, total
from s2i.shared import np_feat
from s2i.within import ALPHAS_WIDE, _blocks

FS, SEG = 200, 200
LAG9 = feat_lag(DEFAULT_LAGS)
VARIANTS = ("clean", "rand_trials20", "rand_trials90", "rand_seg80", "mono_orig", "mono_orig_seg80",
            "chrono90", "rand_trials80", "mono_car", "bip_orig")
# one factor changed at a time (paired over patients): (label, variant A, metric A, variant B, metric B)
ONE_FACTOR = [
    ("split: random vs chronological (20 % calibration)", "rand_trials20", "r_concat", "clean", "r_concat"),
    ("amount: 90 % vs 20 % training, both chronological", "chrono90", "r_concat", "clean", "r_concat"),
    ("split: random vs chronological (90 % training)", "rand_trials90", "r_concat", "chrono90", "r_concat"),
    ("units: random 1-s segments vs random trials (80 %)", "rand_seg80", "r_concat", "rand_trials80", "r_concat"),
    ("target: monopolar vs bipolar (CAR scalp, clean split)", "mono_car", "r_concat", "clean", "r_concat"),
    ("scalp reference: original vs CAR (bipolar target, clean split)", "bip_orig", "r_concat", "clean", "r_concat"),
    ("target + reference: monopolar/original vs bipolar/CAR", "mono_orig", "r_concat", "clean", "r_concat"),
    ("metric: NeuroFlowNet-style vs ours (same predictions, clean)", "clean", "r_nfn", "clean", "r_concat"),
]
METRICS = ("r_concat", "r_window", "r_nfn", "r_nfn_null")


def load(path):
    d = torch.load(path, weights_only=False)
    g = lambda k: d[k].half().float().numpy()                                  # noqa: E731
    return d["trial_bounds"].numpy(), {"car": g("eeg_car"), "orig": g("eeg_orig")}, \
        {"bip": g("ieeg_bip"), "mono": g("ieeg_mono")}


def split(variant, lens, rng):
    """-> (train units, test units, folds); a unit = (trial, start, stop)."""
    n = len(lens)
    trial_unit = lambda i: (i, 0, lens[i])                                     # noqa: E731
    if variant == "chrono90":
        n_tr = max(3, int(round(n * 0.9)))
        folds = [[trial_unit(i) for i in blk] for blk in _blocks(n_tr, 5)]
        return [trial_unit(i) for i in range(n_tr)], [trial_unit(i) for i in range(n_tr + 1, n)], folds
    if variant in ("clean", "mono_orig", "mono_car", "bip_orig"):
        n_c = max(3, int(round(n * 0.2)))
        folds = [[trial_unit(i) for i in blk] for blk in _blocks(n_c, 5)]
        return [trial_unit(i) for i in range(n_c)], [trial_unit(i) for i in range(n_c + 1, n)], folds
    if variant in ("rand_trials20", "rand_trials80", "rand_trials90"):
        n_tr = max(3, int(round(n * {"rand_trials20": 0.2, "rand_trials80": 0.8, "rand_trials90": 0.9}[variant])))
        perm = rng.permutation(n)
        tr, te = sorted(perm[:n_tr]), sorted(perm[n_tr:])
        folds = [[trial_unit(i) for i in sorted(f)] for f in np.array_split(rng.permutation(tr), 5)]
        return [trial_unit(i) for i in tr], [trial_unit(i) for i in te], folds
    segs = [(i, k * SEG, (k + 1) * SEG) for i in range(n) for k in range(lens[i] // SEG)]
    perm = rng.permutation(len(segs))
    n_tr = int(round(0.8 * len(segs)))
    tr = [segs[j] for j in sorted(perm[:n_tr])]
    te = [segs[j] for j in sorted(perm[n_tr:])]
    folds = [[tr[j] for j in sorted(f)] for f in np.array_split(rng.permutation(len(tr)), 5)]
    return tr, te, folds


def by_trial(units):
    out = {}
    for i, s0, s1 in units:
        out.setdefault(i, []).append((s0, s1))
    return out


def run_variant(feat, alphas, E, Y, train, test, folds):
    D, K = feat(E[train[0][0]]).shape[0], Y[0].shape[0]
    fold_of = {u: fi for fi, f in enumerate(folds) for u in f}
    st = [St(D, K) for _ in folds]
    for i, segs in by_trial(train).items():
        X = feat(E[i]).astype(np.float32)
        for s0, s1 in segs:
            st[fold_of[(i, s0, s1)]].add(X[:, s0:s1], Y[i][:, s0:s1])
    tot = total(st)
    m = Fit(lambda tr: ridge_models(tr, alphas), [(tot - f, f) for f in st], tot)
    cv = float(np.nanmean(r_of(m.oof[m.best])))
    P, T = [], []
    for i, segs in by_trial(test).items():
        pred = (feat(E[i]).astype(np.float64).T @ m.A + m.c).T
        for s0, s1 in segs:
            P.append(pred[:, s0:s1].astype(np.float32))
            T.append(Y[i][:, s0:s1])
    return P, T, cv


def windows(P, T):
    for p, t in zip(P, T):
        for k in range(p.shape[1] // SEG):
            yield p[:, k * SEG:(k + 1) * SEG], t[:, k * SEG:(k + 1) * SEG]


def metrics(P, T, rng):
    out = {"r_concat": float(np.nanmean(per_contact_metrics(P, T, FS, bands=False)["r"]))}
    rw, nfn, null = [], [], []
    for p, t in windows(P, T):
        pc, tc = p - p.mean(1, keepdims=True), t - t.mean(1, keepdims=True)
        den = np.sqrt((pc * pc).sum(1) * (tc * tc).sum(1))
        rw.append(np.where(den > 0, (pc * tc).sum(1) / np.maximum(den, 1e-12), np.nan))
        for q, store in ((p, nfn), (rng.normal(size=p.shape), null)):
            sd = q.std(1, keepdims=True)
            ok = (sd[:, 0] > 0) & (t.std(1) > 0)
            if ok.sum() < 2:
                continue
            z = (q[ok] - q[ok].mean(1, keepdims=True)) / sd[ok]
            d = z * t[ok].std(1, keepdims=True) + t[ok].mean(1, keepdims=True)
            store.append(np.corrcoef(d.ravel(), t[ok].ravel())[0, 1])
    out["r_window"] = float(np.nanmean(np.concatenate(rw))) if rw else float("nan")
    out["r_nfn"] = float(np.nanmean(nfn)) if nfn else float("nan")
    out["r_nfn_null"] = float(np.nanmean(null)) if null else float("nan")
    return out


REFS = {"mono_orig": ("orig", "mono"), "mono_orig_seg80": ("orig", "mono"), "mono_car": ("car", "mono"),
        "bip_orig": ("orig", "bip")}


def analyze(path, enc, dev, log, seed, res=None):
    bounds, eegs, tgts = load(path)
    lens = [int(s1 - s0) for s0, s1 in bounds]
    amp = dev.type == "cuda"
    models = {"base": (LAG9, ALPHAS), "enc": (np_feat(enc, dev, amp), ALPHAS_WIDE)}
    res = dict(res or {})
    for v in VARIANTS:
        if v in res:                                   # resume: only the variants still missing
            continue
        rng = np.random.default_rng(seed)
        train, test, folds = split(v, lens, rng)
        ref, tg = REFS.get(v, ("car", "bip"))
        E = [eegs[ref][:, s0:s1] for s0, s1 in bounds]
        Y = [tgts[tg][:, s0:s1] for s0, s1 in bounds]
        res[v] = {}
        for name, (feat, alphas) in models.items():
            t0 = time.time()
            P, T, cv = run_variant(feat, alphas, E, Y, train, test, folds)
            res[v][name] = {**metrics(P, T, np.random.default_rng(seed + 1)), "cv_r": cv}
            log(f"    {v:16s} {name:4s} " + " ".join(f"{k}={res[v][name][k]:+.3f}" for k in METRICS + ("cv_r",)) +
                f" | {time.time() - t0:.0f}s")
    return res


def summarize(results, out_dir, log):
    subs = sorted(results)
    lines = ["| protocol | metric | base | enc | enc / clean enc |", "|---|---|---|---|---|"]
    clean_enc = np.mean([results[s]["clean"]["enc"]["r_concat"] for s in subs])
    summ = {}
    for v in VARIANTS:
        for mt in METRICS:
            row = {m: float(np.nanmean([results[s][v][m][mt] for s in subs])) for m in ("base", "enc")}
            summ[f"{v}/{mt}"] = row
            lines.append(f"| {v} | {mt} | {row['base']:.3f} | {row['enc']:.3f} | {row['enc'] / clean_enc:.2f}x |")
    # literature-style recipes
    lines += ["", "| recipe | n patients | base | enc | enc / clean enc |", "|---|---|---|---|---|"]

    def recipe(name, v, mt, select=False):
        ss = [s for s in subs if not select or results[s][v]["enc"]["cv_r"] > 0.2]
        vals = {m: float(np.nanmean([results[s][v][m][mt] for s in ss])) if ss else float("nan") for m in ("base", "enc")}
        summ[f"recipe/{name}"] = {**vals, "n": len(ss)}
        lines.append(f"| {name} | {len(ss)} | {vals['base']:.3f} | {vals['enc']:.3f} | {vals['enc'] / clean_enc:.2f}x |")
    recipe("ours (clean, r_concat, all patients)", "clean", "r_concat")
    recipe("CAST-like (monopolar, orig ref, random segments, viable patients)", "mono_orig_seg80", "r_concat", True)
    recipe("NeuroFlowNet-like (random 90 % trials, target de-normalisation, flattened r)", "rand_trials90", "r_nfn")
    recipe("NeuroFlowNet metric on RANDOM predictions", "rand_trials90", "r_nfn_null")
    from scipy.stats import wilcoxon
    rng = np.random.default_rng(0)
    lines += ["", "### one factor changed at a time (paired over patients; 95 % CI = patient bootstrap)",
              "| comparison | model | A | B | A − B | 95 % CI | A > B | Wilcoxon p |", "|---|---|---|---|---|---|---|---|"]
    for label, va, ma, vb, mb in ONE_FACTOR:
        for m in ("base", "enc"):
            x = np.array([results[s][va][m][ma] for s in subs])
            y = np.array([results[s][vb][m][mb] for s in subs])
            d = x - y
            boot = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(10000)]
            try:
                p = float(wilcoxon(x, y).pvalue)
            except ValueError:
                p = float("nan")
            summ[f"one_factor/{label}/{m}"] = {"diff": float(d.mean()), "ci": [float(np.percentile(boot, 2.5)),
                                               float(np.percentile(boot, 97.5))], "p": p, "wins": int((d > 0).sum())}
            lines.append(f"| {label} | {m} | {x.mean():.3f} | {y.mean():.3f} | {d.mean():+.3f} | "
                         f"[{np.percentile(boot, 2.5):+.3f}, {np.percentile(boot, 97.5):+.3f}] | "
                         f"{(d > 0).sum()}/{len(d)} | {p:.4f} |")
    sel = [s for s in subs if results[s]["clean"]["enc"]["cv_r"] > 0.2]
    lines.append(f"\nPatient selection on the clean protocol (calibration K-fold r > 0.2): {len(sel)}/{len(subs)} "
                 f"patients, enc r = {np.mean([results[s]['clean']['enc']['r_concat'] for s in sel]):.3f} "
                 f"vs {np.mean([results[s]['clean']['enc']['r_concat'] for s in subs]):.3f} on all patients.")
    lines.append("Note: chrono90 is tested on the last 10 % of the trials (vs the last 80 % for clean), so the "
                 "'amount' comparison also changes the test period.")
    md = "\n".join(lines)
    log("\n" + md)
    with open(os.path.join(out_dir, "protocol.md"), "w", encoding="utf-8") as fh:
        fh.write(md + "\n")
    with open(os.path.join(out_dir, "protocol.json"), "w") as fh:
        json.dump(summ, fh, indent=1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--run", required=True, help="run with enc_<sub>.pt (encoders trained without that patient)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    a = ap.parse_args()
    if sys.platform == "win32":
        sys.stdout.reconfigure(encoding="utf-8")
    os.makedirs(a.out, exist_ok=True)
    log_f = open(os.path.join(a.out, "protocol.log"), "a", encoding="utf-8")

    def log(msg):
        line = f"{time.strftime('%H:%M:%S')} | {msg}"
        print(line, flush=True)
        log_f.write(line + "\n")
        log_f.flush()

    dev = torch.device(a.device)
    torch.set_grad_enabled(False)
    log(f"config: {vars(a)}")
    results = {}
    for ep in sorted(glob.glob(os.path.join(a.run, "enc_sub-*.pt"))):
        sid = os.path.basename(ep)[4:-3]
        fp = os.path.join(a.out, f"protocol_{sid}.json")
        prev = json.load(open(fp)) if os.path.exists(fp) else {}
        if all(v in prev for v in VARIANTS):
            results[sid] = prev
            log(f"[{sid}] already done, loaded")
            continue
        t0 = time.time()
        log(f"[{sid}]" + (f" adding {[v for v in VARIANTS if v not in prev]}" if prev else ""))
        r = analyze(os.path.join(a.data, f"{sid}.pt"), load_encoder(ep, dev), dev, log, a.seed, prev)
        with open(fp + ".tmp", "w") as fh:
            json.dump(r, fh)
        os.replace(fp + ".tmp", fp)
        results[sid] = r
        log(f"[{sid}] done in {(time.time() - t0) / 60:.1f} min")
    if results:
        summarize(results, a.out, log)


if __name__ == "__main__":
    main()

In [ ]:
#@title 2b. Kiểm tra code mới (cả hai số phải > 0)
!grep -c "explicit_frac" {CODE_DIR}/s2i/readout.py
!grep -c "r_nfn_null" {CODE_DIR}/s2i/protocol.py

## 5A. Đánh giá cuối

In [ ]:
#@title 5A-1. Chạy (thêm dự đoán trực tiếp ở calib 20%)
!python -m s2i.readout --data {DATA_DIR} --run {MAIN_RUN} --only_combo --seed_runs {' '.join(SEED_RUNS)} --explicit --out_name readout_final

In [ ]:
#@title 5A-2. Bảng metric + kiểm định + độ phủ
import json, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import wilcoxon
FIN = f'{MAIN_RUN}/readout_final'
R = [json.load(open(f)) for f in sorted(glob.glob(f'{FIN}/readout_sub-*.json'))]
keys = ['r', 'r2', 'r_delta', 'r_theta', 'r_alpha', 'r_beta', 'r_gamma', 'env_alpha', 'env_beta', 'r_no50', 'r_induced']
tab = pd.DataFrame({m: {k: np.nanmean([r['explicit'][m]['mean'].get(k, np.nan) for r in R]) for k in keys}
                    for m in ('base', 'enc', 'combo')}).T
print(f'Calib 20% ({len(R)} bệnh nhân), trung bình theo bệnh nhân:'); display(tab.round(3))
rows = []
for k in ('r', 'r_no50', 'r_induced', 'r2'):
    c = np.array([r['explicit']['combo']['mean'][k] for r in R])
    for ref in ('base', 'enc'):
        b = np.array([r['explicit'][ref]['mean'][k] for r in R])
        rows.append({'metric': k, 'so với': ref, 'combo − ref': (c - b).mean(), 'thắng': f'{(c > b).sum()}/{len(c)}',
                     'Wilcoxon p': wilcoxon(c, b).pvalue})
display(pd.DataFrame(rows).round(4))

# độ phủ - độ chính xác: lọc contact theo r K-fold trên calib (combo)
cv = np.concatenate([r['fracs']['0.2']['models']['combo']['cv_contacts'] for r in R])
te = np.concatenate([r['fracs']['0.2']['models']['combo']['contacts'] for r in R])
ok = np.isfinite(cv) & np.isfinite(te); cv, te = cv[ok], te[ok]
cov = []
for th in (-1, 0.1, 0.2, 0.3, 0.4, 0.5):
    m = cv > th
    cov.append({'ngưỡng r K-fold': th if th > -1 else 'không lọc', '% contact giữ lại': 100 * m.mean(),
                'r test trung bình': te[m].mean() if m.any() else np.nan,
                '% có r test > 0,3': 100 * (te[m] > 0.3).mean() if m.any() else np.nan})
print(f'Độ phủ – độ chính xác (gộp {len(te)} contact):'); display(pd.DataFrame(cov).round(3))
fig, ax = plt.subplots(figsize=(6, 4))
ths = np.linspace(-0.05, 0.6, 40)
ax.plot([100 * (cv > t).mean() for t in ths], [te[cv > t].mean() if (cv > t).any() else np.nan for t in ths], 'o-', ms=3)
ax.set_xlabel('% contact giữ lại (lọc theo r K-fold trên calib)'); ax.set_ylabel('r test trung bình'); ax.grid(alpha=0.3)
ax.set_title('Tái tạo có chọn lọc'); ax.invert_xaxis()
plt.tight_layout(); plt.savefig(f'{FIN}/coverage.png', dpi=130); plt.show()

## 5B. Phân tích protocol

In [ ]:
#@title 5B-1. Chạy
PROT_OUT = f'{RUNS_DIR}/protocol'
!python -m s2i.protocol --data {DATA_DIR} --run {MAIN_RUN} --out {PROT_OUT}

In [ ]:
#@title 5B-2. Kết quả
import json, numpy as np, matplotlib.pyplot as plt
from IPython.display import Markdown, display
display(Markdown(open(f'{PROT_OUT}/protocol.md', encoding='utf-8').read()))
S = json.load(open(f'{PROT_OUT}/protocol.json'))
rec = [k for k in S if k.startswith('recipe/')]
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(rec))
for i, m in enumerate(('base', 'enc')):
    ax.bar(x + i * 0.4, [S[k][m] for k in rec], 0.4, label=m)
ax.set_xticks(x + 0.2); ax.set_xticklabels([k[7:].split(' (')[0] for k in rec], rotation=15)
ax.set_ylabel('r báo cáo'); ax.legend(); ax.grid(alpha=0.3, axis='y')
ax.set_title('Cùng dữ liệu, cùng model, khác protocol')
plt.tight_layout(); plt.savefig(f'{PROT_OUT}/protocol.png', dpi=130); plt.show()

## Cách đọc kết quả

**5A**
- `combo` phải hơn `base` và `enc` ở cả `r`, `r_no50` và `r_induced`. Như vậy các đối chứng đã làm cho `enc` (50 Hz, ERP) cũng đúng với model cuối.
- Bảng độ phủ là bảng "tái tạo có chọn lọc" cho bài. Ví dụ, nếu lọc ngưỡng 0,3 giữ lại khoảng 20% contact với r test khoảng 0,4, ta có thể viết: *"với 20% contact được model đánh giá là đáng tin, r đạt 0,4"*.

**5B**
| Quan sát | Ý nghĩa cho bài |
|---|---|
| `mono_orig` ≫ `clean` | Nhiễm tham chiếu chung thổi phồng r; giải thích con số của CAST |
| `rand_*` > `clean` | Split ngẫu nhiên hoặc cửa sổ kề nhau thổi phồng r (nội suy thay vì dự đoán tương lai) |
| `r_nfn_null` ≫ 0 | Metric kiểu NeuroFlowNet cho r > 0 cả với dự đoán ngẫu nhiên |
| Công thức kiểu CAST hoặc NeuroFlowNet đạt khoảng 0,4–0,6 với **chính model của chúng ta** | Khoảng cách với tài liệu là do protocol, không phải do model yếu: lập luận chính của bài |

Gửi lại: các bảng ở 5A-2 và 5B-2.